## Silver EDA (Deliverables 1.2.2 and 1.2.3)

This notebook performs data profiling, exploratory analysis, and feature behavior review on the Silver pump dataset. It focuses on comparing normal and abnormal operating behavior to understand how the sensors change around failure periods.

The key goals of this notebook are:

- To document data profiling and exploratory analysis for the Silver layer.
- To analyze feature behavior across normal and abnormal operating patterns.
- To generate supporting Silver-layer artifacts (tables, profiles, and charts) that help justify the later model design in the Gold layer.

Outputs from this notebook support the project write-up in Section C by:

- Providing evidence for the anomaly-screening approach described in C.2 and C.2.A.
- Supplying behavior profiles and feature effect-size information that are used to design the Stage 3 rule/profile/historical confirmation layer described in C.2.
- Providing visualizations that can be used in C.6 to communicate how sensor behavior differs between normal and abnormal operation.

## Silver EDA Setup and Imports

In this section I am loading the libraries and project utilities needed for the Silver EDA stage.

The goal here is to get the notebook ready before I begin profiling and comparing the Silver dataset. That includes:
- standard Python libraries
- plotting libraries
- statistical helpers
- clustering and PCA tools
- project path and config utilities
- logging
- truth-record helpers
- artifact saving utilities
- experiment tracking support

At this point I am not analyzing the sensor behavior yet. I am just setting up the notebook so the rest of the exploratory work can run in a structured and repeatable way.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from scipy.stats import skew as scipy_skew, kurtosis as scipy_kurtosis


import pyarrow.parquet as pq
import pyarrow as pa

import json 
import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging, log_layer_paths
from utils.wandb_utils import finalize_wandb_stage

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.postgres_util import get_engine_from_env
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe


# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)



----

## Define a Simple Z-Score Helper for Plotting

This helper function standardizes a numeric series into z-score form.

I use this mainly for plotting and comparison work later in the notebook. The point is not to permanently transform the Silver dataset here. The point is to make different sensor series easier to compare on the same chart when their raw scales are very different.

In [ ]:
def z_score(series: pd.Series) -> pd.Series:
    """
    Z-score normalize a numeric pandas Series.
    Returns a Series with the same index.
    If std is 0 or NaN (constant series), returns series - mean.
    """
    # Ensure Series is a float type
    series = series.astype(float)

    # Use nan-safe stats in case any NaNs sneak in
    mean_value = np.nanmean(series.to_numpy())
    std_value = np.nanstd(series.to_numpy())

    
    if std_value == 0 or np.isnan(std_value):
        # Make all values (effectively) the same or center only
        return pd.Series(
            np.where(series.notna(), 0.0, np.nan),
            index=series.index
        )
    
    return (series - mean_value) / std_value

---

## Load Paths, Configuration, and Runtime Settings

Here I load the resolved paths and configuration values that control how the Silver EDA notebook runs.

This step defines the main runtime context for the notebook, including:
- project folders
- Silver EDA version information
- cleaning recipe ID
- truth-store locations
- dataset identity placeholders
- artifact output folders
- runtime settings for plotting and analysis

I like doing this near the top because it keeps the rest of the notebook cleaner. Instead of hard-coding file paths and settings throughout the workflow, I define them once here and reuse them later.

In [ ]:
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="silver_eda",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

SILVER_EDA_CFG = CONFIG["silver_eda"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# ---- Stage details ----
STAGE = "silver_eda"
LAYER_NAME = SILVER_EDA_CFG["layer_name"]
SILVER_VERSION = CONFIG["versions"]["silver_eda"]
CLEANING_RECIPE_ID = SILVER_EDA_CFG["cleaning_recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]

PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"]

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = None

SILVER_TRUTH_HASH = None
SILVER_PARENT_TRUTH_HASH = None
SILVER_TRUTH_PATH = None
SILVER_PARENT_TRUTH_PATH = None
SILVER_PARENT_LAYER_NAME = "silver"

LABEL_SOURCE_COLUMN = None
LABEL_SOURCE_TYPE = None
LABEL_SOURCE_INFO = {}
CANONICAL_INFO = {}
FEATURE_SET_INFO = {}
QUALITY_INFO = {}
NEEDS_ONE_HOT_ENCODING = None
ONE_HOT_ENCODING_COLUMNS = []

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

SILVER_PROCESS_RUN_ID = make_process_run_id(SILVER_EDA_CFG["process_run_id_prefix"])

# ---- W&B ----
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{SILVER_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# Thresholds
MIN_TIME_PARSE_SUCCESS_PERCENT = float(SILVER_EDA_CFG["min_time_parse_success_percent"])
MIN_STEP_PARSE_SUCCESS_PERCENT = float(SILVER_EDA_CFG["min_step_parse_success_percent"])

QUARANTINE_MISSING_PCT = float(SILVER_EDA_CFG["quarantine_missing_pct"])
CORRELATION_THRESHOLD = float(SILVER_EDA_CFG["correlation_threshold"])

USE_ROBUST_SCALER = bool(SILVER_EDA_CFG["use_robust_scaler"])

TOP_N_SENSORS_FOR_PLOTS = int(SILVER_EDA_CFG["top_n_sensors_for_plots"])
PAIRPLOT_SENSOR_CAP = int(SILVER_EDA_CFG["pairplot_sensor_cap"])
PAIRPLOT_SAMPLE_N = int(SILVER_EDA_CFG["pairplot_sample_n"])
TOP_PLOT_COLS = int(SILVER_EDA_CFG["top_plot_cols"])
TOP_CORR_COLS = int(SILVER_EDA_CFG["top_corr_cols"])

ROLLING_MINUTES = int(SILVER_EDA_CFG["rolling_minutes"])
LOOKBACK_HOURS = int(SILVER_EDA_CFG["lookback_hours"])
BASELINE_DAYS = int(SILVER_EDA_CFG["baseline_days"])
BASELINE_GAP_HOURS = int(SILVER_EDA_CFG["baseline_gap_hours"])
SUSTAIN_MINUTES = int(SILVER_EDA_CFG["sustain_minutes"])
TOP_SENSOR_PRE_HOURS = int(SILVER_EDA_CFG["top_sensor_pre_hours"])

PRE_WINDOW_STEPS = int(SILVER_EDA_CFG["pre_window_steps"])
POST_WINDOW_STEPS = int(SILVER_EDA_CFG["post_window_steps"])
MAX_ONSETS_TO_USE = int(SILVER_EDA_CFG["max_onsets_to_use"])
PCA_SAMPLE_ROW_COUNT = int(SILVER_EDA_CFG["pca_sample_row_count"])
IMPUTE_SAMPLE_ROW_COUNT = int(SILVER_EDA_CFG["impute_sample_row_count"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

# ---- File names ----
BRONZE_TRAIN_DATA_FILE_NAME = FILENAMES["bronze_train_file_name"]
SILVER_TRAIN_DATA_FILE_NAME = FILENAMES["silver_train_file_name"]

FEATURE_REGISTRY_FILE_NAME = None
FEATURE_REGISTRY_PATH = None

# ---- Paths setup ----
BRONZE_TRAIN_DATA_PATH = Path(PATHS["data_bronze_train_dir"])
SILVER_TRAIN_DATA_PATH = Path(PATHS["data_silver_train_dir"])

SILVER_ARTIFACTS_PATH = Path(PATHS["artifacts_root"]) / "silver"
SILVER_EDA_ARTIFACTS_ROOT = Path(PATHS["artifacts_root"]) / "silver_eda"
SILVER_EDA_ARTIFACTS_PATH = None

ARTIFACTS_PATH = Path(PATHS["artifacts_root"])

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

LOGS_PATH = Path(PATHS["logs_root"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

required_columns = [
    "anomaly_flag",
    "event_step",
    "time_index",
    "meta__asset_id",
    "meta__run_id",
]

set_wandb_dir_from_config(CONFIG)

SILVER_TRAIN_DATA_PATH.mkdir(parents=True, exist_ok=True)
SILVER_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
SILVER_EDA_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

----

## Start Logging for the Silver EDA Stage

Before I load the data, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, traceability, and project discipline. If I need to confirm which input file was used or where a later step failed, the log gives me a cleaner record than notebook output alone.

In [ ]:
# Logging Setup

# Create silver log path 
silver_log_path = paths.logs / "silver_eda.log"

# Initial Logger
configure_logging(
    "capstone",
    silver_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.silver_eda")

# Log load and initiation
logger.info("Silver EDA stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="silver", logger=logger)


----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the Silver EDA stage.

I am using this mainly for run tracking and artifact registration. It helps document the notebook settings and saved outputs, but it does not change the exploratory logic itself.

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="silver_eda",
    config={
        "silver_version": SILVER_VERSION,
        "cleaning_recipe_id": CLEANING_RECIPE_ID,
        "quarantine_missing_pct": QUARANTINE_MISSING_PCT,
        "min_time_parse_success_percent": MIN_TIME_PARSE_SUCCESS_PERCENT,
        "rolling_window": ROLLING_MINUTES,
        "silver_path": str(SILVER_TRAIN_DATA_PATH / SILVER_TRAIN_DATA_FILE_NAME),
        "silver_out_dir": str(SILVER_TRAIN_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


----

## Initialize the Silver EDA Ledger

Here I create the ledger that tracks the main steps taken during the Silver EDA notebook.

I treat the ledger as a structured run record. It gives me a cleaner summary of the exploratory workflow than relying only on printed notebook output, especially when I need to review or reference the notebook later.

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=CLEANING_RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


---

## Load the Silver Dataset for Exploratory Review

This is the point where I load the Silver dataset that came out of the earlier Silver preprocessing work.

The purpose of this notebook is not to rebuild Silver preprocessing. It is to inspect the resulting Silver data more closely so I can understand:
- overall structure
- missingness
- duplicate risk
- state behavior
- feature movement between normal and abnormal periods
- correlation and grouping patterns
- artifacts that may support later Gold modeling decisions

In [ ]:

# Load Data

preferred_silver = SILVER_TRAIN_DATA_PATH / SILVER_TRAIN_DATA_FILE_NAME

if preferred_silver.exists():
    silver_data_path = preferred_silver
else:
    parquet_files = sorted(SILVER_TRAIN_DATA_PATH.glob("*.parquet"))
    if len(parquet_files) == 0:
        raise FileNotFoundError(f"No parquet files found in {SILVER_TRAIN_DATA_PATH}")
    if len(parquet_files) > 1: 
        logger.warning("Multiple Parquet Files found; Using First %s", parquet_files[0])
    silver_data_path = parquet_files[0]

if not silver_data_path.exists():
    raise FileNotFoundError(f"Silver parquet not found: {silver_data_path}")
    
dataframe = load_data(silver_data_path.parent, silver_data_path.name)



#### #### #### #### #### #### #### #### 

logger.info("Loaded Silver: %s | shape=%s", silver_data_path, dataframe.shape)
wandb_run.log({"silver_rows": int(dataframe.shape[0]), "silver_cols": int(dataframe.shape[1])})

ledger.add(
    kind="step",
    step="load_silver",
    message="Loaded Silver Parquet",
    why="Silver must be derived from reprodicible Silver Artifact",
    consequence="All silver outputs trace back to this file",
    data={"silver_path": str(silver_data_path), "shape": list(dataframe.shape), "cols": len(dataframe.columns)},
    logger=logger
)


#### #### #### #### #### #### #### #### 

display(dataframe.head(3))

----

## Initial Structural Review of the Silver Dataset

Before I start producing deeper artifacts, I want a first look at the Silver dataframe as it enters this notebook.

At this stage I am checking:
- shape
- column types
- general dataframe structure
- a quick numeric summary

This is not the deep interpretation step yet. It is more of a first trust check to make sure the Silver handoff looks usable before I move into profiling and EDA.

In [ ]:
print("Silver training shape:", dataframe.shape)
dataframe.info()
display(dataframe.describe().T.head(10))



### Ask

What is this first review helping me confirm?

### Answer

This check helps me answer a basic question: **did the Silver dataset arrive in a stable and usable form for exploratory analysis?**

At this point I am mainly looking for broad structural confirmation:
- the row and column counts make sense
- the dtypes do not look obviously broken
- the dataframe appears complete enough to analyze
- the Silver handoff looks consistent with what the earlier notebook should have produced

So this is more of a readiness check than a true interpretation step.

----

## Resolve the Silver Truth Record and Confirm Dataset Identity

Before I start building EDA artifacts, I want to confirm which Silver truth record this dataframe came from and what dataset identity it carries.

This matters because the EDA notebook should stay tied to the saved Silver output rather than treating the dataframe like an untracked standalone table. In this step I confirm:
- the Silver truth hash
- the dataset name
- the parent truth relationship
- the saved artifact references that this notebook may need later

In [ ]:
SILVER_TRUTH_HASH = extract_truth_hash(dataframe)

if SILVER_TRUTH_HASH is None:
    raise ValueError("Could not resolve meta__truth_hash from Silver dataframe.")

SILVER_DATASET_NAME = (
    dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
SILVER_DATASET_NAME = SILVER_DATASET_NAME[SILVER_DATASET_NAME != ""]

if len(SILVER_DATASET_NAME) == 0:
    raise ValueError("Silver dataframe is missing usable meta__dataset values.")

SILVER_DATASET_NAME = str(SILVER_DATASET_NAME.iloc[0]).strip()

silver_truth = load_parent_truth_record_from_dataframe(
    dataframe=dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="silver",
    dataset_name=SILVER_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(silver_truth)
SILVER_TRUTH_HASH = get_truth_hash(silver_truth)
SILVER_PARENT_TRUTH_HASH = get_parent_truth_hash(silver_truth)

PIPELINE_MODE_FROM_TRUTH = get_pipeline_mode_from_truth(silver_truth)
if PIPELINE_MODE_FROM_TRUTH is not None:
    PIPELINE_MODE = PIPELINE_MODE_FROM_TRUTH

SILVER_TRUTH_PATH = (
    TRUTHS_PATH
    / "silver"
    / f"{DATASET_NAME}__silver__truth__{SILVER_TRUTH_HASH}.json"
)

LABEL_SOURCE_COLUMN = get_truth_value(
    silver_truth,
    "runtime_facts",
    "label_resolution",
    "label_source_column",
    required=False,
)

LABEL_SOURCE_TYPE = get_truth_value(
    silver_truth,
    "runtime_facts",
    "label_resolution",
    "label_source_type",
    required=False,
)

CANONICAL_INFO = silver_truth.get("runtime_facts", {}).get("canonical_info", {})
FEATURE_SET_INFO = silver_truth.get("runtime_facts", {}).get("feature_set", {})
QUALITY_INFO = silver_truth.get("runtime_facts", {}).get("quality_info", {})

NEEDS_ONE_HOT_ENCODING = bool(
    silver_truth.get("needs_one_hot_encoding", False)
)
ONE_HOT_ENCODING_COLUMNS = list(
    silver_truth.get("one_hot_encoding_columns", [])
)

feature_registry_dir = get_artifact_path_from_truth(
    silver_truth,
    "feature_registry_dir",
)
FEATURE_REGISTRY_FILE_NAME = f"{DATASET_NAME}__silver__feature_registry.json"
FEATURE_REGISTRY_PATH = Path(feature_registry_dir) / FEATURE_REGISTRY_FILE_NAME

SILVER_EDA_ARTIFACTS_PATH = SILVER_EDA_ARTIFACTS_ROOT / DATASET_NAME
SILVER_EDA_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### 

logger.info("Loaded Silver truth: %s", SILVER_TRUTH_PATH)
logger.info("Resolved Silver EDA dataset name from Silver truth: %s", DATASET_NAME)
logger.info("Resolved label source column from Silver truth: %s", LABEL_SOURCE_COLUMN)

print("Loaded Silver truth:", SILVER_TRUTH_PATH)
print("Silver truth hash:", SILVER_TRUTH_HASH)
print("Resolved Silver EDA dataset name:", DATASET_NAME)
print("Resolved label source column:", LABEL_SOURCE_COLUMN)

#### #### #### #### #### #### #### #### 

---

## Load the Silver Feature Registry

Now I load the feature registry created earlier in the Silver pipeline.

This is important because the EDA notebook should not guess which columns are the core features. It should use the saved feature registry so the profiling and comparisons stay aligned with the selected Silver feature set.

In [ ]:
if FEATURE_REGISTRY_PATH is None:
    raise ValueError("FEATURE_REGISTRY_PATH was not resolved from Silver truth before loading the feature registry.")

feature_registry = load_json(FEATURE_REGISTRY_PATH.parent, FEATURE_REGISTRY_PATH.name)

FEATURE_COLUMNS = feature_registry.get("feature_columns", [])


#### #### #### #### #### #### #### #### 

logger.info("Loaded Silver Feature Registry: %s", FEATURE_REGISTRY_PATH)
wandb_run.log({"feature_registry_keys": int(len(feature_registry))})

ledger.add(
    kind="step",
    step="load_silver_feature_registry",
    message="Loaded Silver Feature Registry JSON file using the path resolved from Silver truth.",
    why="Silver EDA should inherit resolved feature metadata from Silver Pre-EDA rather than rebuilding it from config.",
    consequence="Silver EDA uses the same resolved feature set and feature registry lineage as Silver Pre-EDA.",
    data={
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "keys": int(len(feature_registry)),
        "feature_count": int(len(FEATURE_COLUMNS)),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

----

## Build a Quick Silver Overview Summary

This section creates a compact high-level overview of the Silver dataset.

The purpose is to save a simple structural snapshot that records:
- row count
- column count
- whether the required columns are present

I like having this as a saved artifact because it gives me one quick reference point for the basic state of the Silver data without needing to rerun the full notebook.

In [ ]:
def build_silver_overview_summary(dataframe: pd.DataFrame) -> dict:
    """
    Build a quick structural overview of the Silver dataframe.
    """
    return {
        "row_count": int(len(dataframe)),
        "column_count": int(len(dataframe.columns)),
        "meta_column_count": int(sum(column.startswith("meta__") for column in dataframe.columns)),
        "numeric_column_count": int(len(dataframe.select_dtypes(include=[np.number]).columns)),
        "categorical_column_count": int(
            len([
                column for column in dataframe.columns
                if not pd.api.types.is_numeric_dtype(dataframe[column])
                and not pd.api.types.is_datetime64_any_dtype(dataframe[column])
            ])
        ),
        "columns_are_unique": bool(dataframe.columns.is_unique),
    }

In [ ]:
silver_overview_summary = build_silver_overview_summary(dataframe)

logger.info("Overview: %s", silver_overview_summary)

save_json(
    silver_overview_summary,
    file_path=SILVER_EDA_ARTIFACTS_PATH,
    file_name="silver_eda__overview.json",
)

wandb.save(str(SILVER_EDA_ARTIFACTS_PATH / "silver_eda__overview.json"))

silver_overview_summary

---

## Audit Missingness Across the Full Silver Dataset

This step profiles missing values across every column in the Silver dataframe.

Missingness matters here for two reasons:
- it helps describe the current quality of the Silver output
- it also helps guide later modeling and synthetic-data decisions

So this is not only a data-cleaning check. It is also part of understanding how stable each field is at this stage of the pipeline.

In [ ]:
def build_missingness_audit_table(
    dataframe: pd.DataFrame,
    *,
    include_only_nonzero: bool = False,
) -> pd.DataFrame:
    """
    Build missingness summary by column.
    """
    rows = []
    total_rows = len(dataframe)

    for column_name in dataframe.columns:
        null_count = int(dataframe[column_name].isna().sum())
        missing_pct = float((null_count / total_rows) * 100) if total_rows > 0 else 0.0

        if include_only_nonzero and null_count == 0:
            continue

        rows.append(
            {
                "column_name": column_name,
                "null_count": null_count,
                "missing_pct": round(missing_pct, 4),
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result

    return result.sort_values(
        ["missing_pct", "column_name"],
        ascending=[False, True],
    ).reset_index(drop=True)

In [ ]:
null_table = build_missingness_audit_table(dataframe)

logger.info("Null Table: %s", null_table.shape)

null_all_path = SILVER_EDA_ARTIFACTS_PATH / "nulls__all_columns.csv"
null_table.to_csv(null_all_path, index=False)
wandb.save(str(null_all_path))

feature_null_table = null_table[
    null_table["column_name"].isin(FEATURE_COLUMNS)
].copy()

logger.info("Feature Null Table: %s", feature_null_table.shape)

feature_null_path = SILVER_EDA_ARTIFACTS_PATH / "nulls__feature_columns.csv"
feature_null_table.to_csv(feature_null_path, index=False)
wandb.save(str(feature_null_path))

null_table.head(15), feature_null_table.head(15)


### Ask

Why does missingness matter this much in the Silver EDA notebook?

### Answer

Because missingness is doing more than just telling me whether the data is incomplete.

In this project, missingness can matter in several ways:
- it may reveal weak or unstable features
- it may show state-specific behavior
- it may influence which features are safe to model
- it may affect how I choose imputation later

So I am not just counting nulls here. I am using missingness to understand which parts of the Silver dataset look strong and which parts need more caution.

---

## Check for Duplicate Rows and Duplicate Event IDs

This section checks whether the Silver dataset contains duplicate rows or duplicate event identifiers.

I want this checkpoint because duplicate rows can distort counts, plots, and later model behavior. Duplicate event IDs are especially important because they can indicate that the event-order structure is not as clean as it should be.

In [ ]:
def build_duplicate_summary(dataframe: pd.DataFrame) -> dict:
    """
    Summarize duplicate rows and duplicate record/event keys.
    """
    duplicate_row_count = int(dataframe.duplicated().sum())

    duplicate_meta_record_id_count = None
    if "meta__record_id" in dataframe.columns:
        duplicate_meta_record_id_count = int(
            dataframe["meta__record_id"].duplicated().sum()
        )

    duplicate_event_id_count = None
    if "event_id" in dataframe.columns:
        duplicate_event_id_count = int(
            dataframe["event_id"].duplicated().sum()
        )

    return {
        "duplicate_row_count": duplicate_row_count,
        "duplicate_meta__record_id_count": duplicate_meta_record_id_count,
        "duplicate_event_id_count": duplicate_event_id_count,
    }

In [ ]:
dup_info = build_duplicate_summary(dataframe)

logger.info("Duplicate Information: %s", dup_info)

save_json(dup_info, file_path=SILVER_EDA_ARTIFACTS_PATH, file_name="duplicates__summary.json")
wandb.save(str(SILVER_EDA_ARTIFACTS_PATH / "duplicates__summary.json"))

dup_info

---

## Profile Numeric and Categorical-Like Columns

Here I create the broader descriptive statistics for the Silver dataset.

This section helps summarize:
- numeric distribution behavior
- basic descriptive statistics
- categorical-like column cardinality

I use this as a more general profiling layer before I move into state-specific behavior and sensor-level comparisons.

In [ ]:
def build_numeric_describe_table(
    dataframe: pd.DataFrame,
    *,
    include_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    Build numeric describe table.
    """
    if include_columns is None:
        numeric_columns = dataframe.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_columns = [
            column for column in include_columns
            if column in dataframe.columns and pd.api.types.is_numeric_dtype(dataframe[column])
        ]

    if len(numeric_columns) == 0:
        return pd.DataFrame()

    describe_df = dataframe[numeric_columns].describe().T.reset_index()
    describe_df = describe_df.rename(columns={"index": "column_name"})
    return describe_df

In [ ]:
numeric_describe = build_numeric_describe_table(dataframe)

numeric_stats_path = SILVER_EDA_ARTIFACTS_PATH / "column_stats__numeric_describe.csv"
numeric_describe.to_csv(numeric_stats_path)

logger.info("Numeric Columns Found: %s", numeric_describe)

wandb.save(str(numeric_stats_path))


In [ ]:
def build_categorical_cardinality_table(
    dataframe: pd.DataFrame,
    *,
    exclude_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    """
    Build categorical cardinality table.
    """
    exclude_columns = set(exclude_columns or [])

    rows = []
    for column_name in dataframe.columns:
        if column_name in exclude_columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[column_name]):
            continue
        if pd.api.types.is_datetime64_any_dtype(dataframe[column_name]):
            continue

        series = dataframe[column_name].astype("string")
        rows.append(
            {
                "column_name": column_name,
                "non_null_count": int(series.notna().sum()),
                "null_count": int(series.isna().sum()),
                "unique_count": int(series.nunique(dropna=True)),
            }
        )

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values(
        ["unique_count", "column_name"],
        ascending=[False, True],
    ).reset_index(drop=True)

In [ ]:

cardinality_table = build_categorical_cardinality_table(dataframe)


logger.info("Categorical Cardinality Columns Found: %s", cardinality_table)

cardinality_path = SILVER_EDA_ARTIFACTS_PATH / "column_stats__categorical_cardinality.csv"
cardinality_table.to_csv(cardinality_path, index=False)
wandb.save(str(cardinality_path))



In [ ]:
numeric_describe.head(10), cardinality_table.head(15)

---

## Resolve the source state column from truth

Before building any state-based analysis, the notebook first resolves which column should be treated as the source state or label field.

This keeps the workflow flexible across datasets and pipeline runs. If the truth record already captured a label source column during an earlier step, that column is used here. Otherwise, the notebook falls back to a default operational state column such as `machine_status`.

This approach avoids hardcoding one column name throughout the notebook and makes the Silver EDA workflow easier to reuse across different datasets.

In [ ]:
def resolve_state_column_from_truth(
    truth_record: dict,
    *,
    fallback_status_column: str = "machine_status",
) -> str:
    """
    Resolve the status/label source column from truth.
    """
    runtime_facts = (truth_record or {}).get("runtime_facts", {}) or {}
    label_resolution = runtime_facts.get("label_resolution", {}) or {}

    label_source_column = label_resolution.get("label_source_column")
    if label_source_column is not None and str(label_source_column).strip() != "":
        return str(label_source_column).strip()

    return fallback_status_column

## Build a normalized working state column

Once the source state column is identified, the notebook creates a normalized working version of that column for Silver EDA.

The goal is to preserve the original source labels while also creating a consistent vocabulary for downstream analysis. In this notebook, raw values such as `broken` and `recovering` are mapped into a simpler normalized state set:

- `normal`
- `abnormal`
- `recovery`

Any values that do not appear in the mapping are retained in cleaned lowercase form so they remain visible for review instead of being silently dropped.

This normalized state column is treated as a Silver EDA working field. It is primarily used to support state-aware profiling and related exploratory analysis.

In [ ]:
def build_state_col_synth(
    dataframe: pd.DataFrame,
    *,
    status_column: str,
    state_map: Optional[dict] = None,
    output_column: Optional[str] = None,
) -> tuple[pd.DataFrame, str]:
    """
    Build normalized synthetic state column.
    """
    if status_column not in dataframe.columns:
        raise KeyError(f"Missing status column: {status_column}")

    working = dataframe.copy()
    output_column = output_column or f"{status_column}__synthetic"

    series = (
        working[status_column]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
    )

    if state_map is not None:
        mapped = series.map(state_map)
        series = mapped.fillna(series)

    working[output_column] = series
    return working, output_column

## Review normalized state coverage

After normalization, the notebook checks both the original and normalized state values to confirm that the mapping behaved as expected.

This step serves two purposes:

1. It validates that the resolved source state column contains the expected labels.
2. It confirms that the normalized working column provides the state groupings needed for downstream analysis.

The notebook also creates simple boolean masks for the major state classes and builds a list of observed normalized states. These will be used in the next section to drive loop-based profile generation.

In [ ]:
FALLBACK_STATE_COL = "machine_status"

state_column = resolve_state_column_from_truth(
    silver_truth,
    fallback_status_column=FALLBACK_STATE_COL,
)

if state_column not in dataframe.columns:
    raise KeyError(
        f"Resolved state_column='{state_column}' not found in dataframe columns."
    )

logger.info("Resolved state column from Silver truth: %s", state_column)
print("Resolved state_column:", state_column)

state_map = {
    "normal": "normal",
    "broken": "abnormal",
    "recovering": "recovery",
}

dataframe, state_column_synthetic = build_state_col_synth(
    dataframe,
    status_column=state_column,
    state_map=state_map,
)

state_normalization_payload = {
    "source_state_column": state_column,
    "normalized_state_column": state_column_synthetic,
    "state_map": state_map,
    "observed_source_states": sorted(
        dataframe[state_column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    ),
    "observed_normalized_states": sorted(
        dataframe[state_column_synthetic]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),
}

logger.info("State normalization payload: %s", state_normalization_payload)
print("State normalization payload:", state_normalization_payload)

print("Original states:", sorted(dataframe[state_column].dropna().astype(str).unique()))
print("Synthetic states:", sorted(dataframe[state_column_synthetic].dropna().unique()))

normal_mask = dataframe[state_column_synthetic].eq("normal")
abnormal_mask = dataframe[state_column_synthetic].eq("abnormal")
recovery_mask = dataframe[state_column_synthetic].eq("recovery")

print(
    "normal:", int(normal_mask.sum()),
    "abnormal:", int(abnormal_mask.sum()),
    "recovery:", int(recovery_mask.sum()),
)

state_list = sorted(
    map(str, dataframe[state_column_synthetic].dropna().unique())
)
print("State list:", state_list)

----

## Define a reusable state profile builder

Rather than writing separate profiling blocks for each state, the notebook uses a reusable helper function that can generate a sensor profile table for any filtered state subset.

For each numeric feature, the function summarizes:

- central tendency
- spread
- percentile bounds
- distribution shape hints
- generation guardrails

This keeps the profiling logic compact, repeatable, and easier to maintain as the notebook moves toward a more dataset-agnostic structure.

In [ ]:
def build_state_sensor_profile_table(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    state_value: str,
    include_missing_pct: bool = False,
    keep_empty_rows: bool = False,
) -> pd.DataFrame:
    """
    Build a sensor profile table for one already-filtered normalized state slice.

    Parameters
    ----------
    dataframe:
        A dataframe already filtered to a single state.
    feature_columns:
        Feature columns to profile.
    state_value:
        The normalized state label to stamp into the output.
    include_missing_pct:
        If True, include missing_pct in the output.
    keep_empty_rows:
        If True, keep rows for features that have no usable numeric values.
    """
    profile_rows = []

    for column_name in feature_columns:
        if column_name not in dataframe.columns:
            continue

        series_raw = (
            pd.to_numeric(dataframe[column_name], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
        )

        missing_pct = float(series_raw.isna().mean() * 100.0)
        series = series_raw.dropna()

        if series.empty:
            if keep_empty_rows:
                row = {
                    "sensor": column_name,
                    "state_scope": state_value,
                    "count": 0,
                    "mean": np.nan,
                    "std": np.nan,
                    "min": np.nan,
                    "max": np.nan,
                    "median": np.nan,
                    "iqr": np.nan,
                    "p01": np.nan,
                    "p05": np.nan,
                    "p25": np.nan,
                    "p50": np.nan,
                    "p75": np.nan,
                    "p95": np.nan,
                    "p99": np.nan,
                    "skewness": np.nan,
                    "kurtosis": np.nan,
                    "robust_std": np.nan,
                    "distribution_family": "no_values",
                    "lower_bound": np.nan,
                    "upper_bound": np.nan,
                }
                if include_missing_pct:
                    row["missing_pct"] = missing_pct
                profile_rows.append(row)
            continue

        p25 = float(series.quantile(0.25))
        p75 = float(series.quantile(0.75))
        iqr = float(p75 - p25)
        robust_std = float(iqr / 1.349) if iqr > 0 else 0.0

        p01 = float(series.quantile(0.01))
        p99 = float(series.quantile(0.99))

        skewness = (
            float(scipy_skew(series.values, bias=False))
            if series.shape[0] > 2
            else np.nan
        )
        kurtosis = (
            float(scipy_kurtosis(series.values, bias=False, fisher=True))
            if series.shape[0] > 3
            else np.nan
        )

        sample_std = float(series.std(ddof=1)) if series.shape[0] > 1 else 0.0

        if sample_std <= 1e-8:
            family = "near_constant"
        elif np.isfinite(skewness) and skewness >= 1.0:
            family = "right_skewed"
        elif np.isfinite(skewness) and skewness <= -1.0:
            family = "left_skewed"
        else:
            family = "bounded_normal"

        row = {
            "sensor": column_name,
            "state_scope": state_value,
            "count": int(series.shape[0]),
            "mean": float(series.mean()),
            "std": sample_std,
            "min": float(series.min()),
            "max": float(series.max()),
            "median": float(series.median()),
            "iqr": iqr,
            "p01": p01,
            "p05": float(series.quantile(0.05)),
            "p25": p25,
            "p50": float(series.quantile(0.50)),
            "p75": p75,
            "p95": float(series.quantile(0.95)),
            "p99": p99,
            "skewness": skewness,
            "kurtosis": kurtosis,
            "robust_std": robust_std,
            "distribution_family": family,
            "lower_bound": p01,
            "upper_bound": p99,
        }

        if include_missing_pct:
            row["missing_pct"] = missing_pct

        profile_rows.append(row)

    if not profile_rows:
        return pd.DataFrame()

    sort_columns = ["std", "sensor"]
    return (
        pd.DataFrame(profile_rows)
        .sort_values(sort_columns, ascending=[False, True])
        .reset_index(drop=True)
    )

## Build and export sensor profiles by normalized state

With the normalized state column in place, the notebook now loops through each observed state and builds a sensor profile table for that state subset.

Each exported profile captures the behavior of the numeric sensor features within a specific operating condition. These profile tables are useful for:

- comparing state-level sensor behavior
- supporting downstream anomaly interpretation
- providing profile inputs for the Synthetic Data Generator

This loop-based approach replaces the need for separate handwritten profiling sections for each state while still allowing the notebook to preview and save each result individually.

In [ ]:
feature_profile_paths = {}

for state_value in state_list:
    working_dataframe = dataframe.loc[
        dataframe[state_column_synthetic] == state_value
    ].copy()

    working_profile_table = build_state_sensor_profile_table(
        working_dataframe,
        feature_columns=FEATURE_COLUMNS,
        state_value=state_value,
        include_missing_pct=False,
        keep_empty_rows=False,
    )

    if working_profile_table.empty:
        print(f"No numeric features produced a profile for {state_value}.")
        continue

    safe_state = re.sub(r"[^A-Za-z0-9._-]+", "_", str(state_value)).strip("_")

    out_name = f"{DATASET_NAME}__{LAYER_NAME}__feature_profile__{safe_state}.csv"
    out_path = SILVER_EDA_ARTIFACTS_PATH / out_name

    working_profile_table.to_csv(out_path, index=False)
    wandb.save(str(out_path))

    feature_profile_paths[safe_state] = str(out_path)

    wandb.log({
        f"profile_preview/{safe_state}": wandb.Table(
            dataframe=working_profile_table.head(50)
        )
    })

    print(f"\nProfile preview for state: {state_value}")
    display(working_profile_table.head(15))

### Ask

Why am I creating a synthetic state mapping if I already have a state column?

### Answer

Because the original state labels are useful, but they may not be in the exact format I want for later profiling and synthetic-data support work.

By mapping them into a simpler normalized set, I make it easier to:
- compare states consistently
- build state-based feature profiles
- create later artifacts that depend on a predictable state vocabulary

So this is mostly a standardization step, not a replacement of the original source meaning.

----

## Revisit the Dropped Sensors from Silver Pre-EDA

This section looks back at the sensors that were quarantined or dropped earlier in Silver Pre-EDA.

I am doing this because dropping a feature in preprocessing does not automatically mean it has no interesting behavior. It just means it was not retained in the final Silver feature set. By profiling the dropped sensors here, I can still document what they looked like and whether they showed any useful state-specific patterns.

In [ ]:
# ---------------------------
# 1) Load Silver Pre-EDA truth and dropped parquet
# ---------------------------

SILVER_PREEDA_TRUTH_HASH = get_truth_hash(silver_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

print("Loaded truth record layer_name:", silver_preeda_truth.get("layer_name"))
print("Loaded truth hash:", silver_preeda_truth.get("truth_hash"))
print("Loaded parent truth hash:", silver_preeda_truth.get("parent_truth_hash"))
print("Artifact paths keys:", list((silver_preeda_truth.get("artifact_paths", {}) or {}).keys()))
print("Runtime facts keys:", list((silver_preeda_truth.get("runtime_facts", {}) or {}).keys()))
print("Sensor drop audit:", (silver_preeda_truth.get("runtime_facts", {}) or {}).get("sensor_drop_audit", {}))

In [ ]:

drop_audit = (silver_preeda_truth.get("runtime_facts", {}) or {}).get("sensor_drop_audit", {}) or {}
dropped_sensors = drop_audit.get("dropped_features", []) or []

dropped_parquet_path = (silver_preeda_truth.get("artifact_paths", {}) or {}).get("silver_preeda_dropped_sensors_parquet")

print("Silver Pre-EDA truth hash:", SILVER_PREEDA_TRUTH_HASH)
print("Dropped sensors:", dropped_sensors)
print("Dropped parquet path:", dropped_parquet_path)

In [ ]:
def load_dropped_sensor_dataframe(
    *,
    dropped_path,
) -> pd.DataFrame:
    """
    Load the dropped-sensor parquet produced by Silver Pre-EDA.
    """
    dropped_path = Path(dropped_path)

    if not dropped_path.exists():
        raise FileNotFoundError(f"Dropped sensor parquet not found: {dropped_path}")

    return pd.read_parquet(dropped_path)

In [ ]:
dropped_df = load_dropped_sensor_dataframe(
    dropped_path=dropped_parquet_path,
)

print("Dropped df shape:", dropped_df.shape)
display(dropped_df.head())

In [ ]:
def attach_state_column_to_dropped_dataframe(
    dropped_dataframe: pd.DataFrame,
    *,
    silver_dataframe: pd.DataFrame,
    state_column: str,
    synthetic_state_column: str,
    join_key: str = "meta__record_id",
) -> pd.DataFrame:
    """
    Attach the resolved raw state column and normalized synthetic state column
    from the Silver EDA dataframe onto the dropped-sensor dataframe.
    """
    if join_key not in dropped_dataframe.columns:
        raise KeyError(f"Dropped dataframe missing join key: {join_key}")

    if join_key not in silver_dataframe.columns:
        raise KeyError(f"Silver dataframe missing join key: {join_key}")

    if state_column not in silver_dataframe.columns:
        raise KeyError(f"Silver dataframe missing state column: {state_column}")

    if synthetic_state_column not in silver_dataframe.columns:
        raise KeyError(
            f"Silver dataframe missing synthetic state column: {synthetic_state_column}"
        )

    right_columns = [join_key, state_column, synthetic_state_column]
    right_frame = (
        silver_dataframe[right_columns]
        .drop_duplicates(subset=[join_key])
        .copy()
    )

    merged = dropped_dataframe.merge(
        right_frame,
        on=join_key,
        how="left",
    )

    return merged

In [ ]:
dropped_df = attach_state_column_to_dropped_dataframe(
    dropped_dataframe=dropped_df,
    silver_dataframe=dataframe,
    state_column=state_column,
    synthetic_state_column=state_column_synthetic,
    join_key="meta__record_id",
)

print("Dropped df with states shape:", dropped_df.shape)
print("Synthetic states:", sorted(dropped_df[state_column_synthetic].dropna().astype(str).unique()))
display(dropped_df.head())

In [ ]:
dropped_present = [
    sensor_name
    for sensor_name in dropped_sensors
    if sensor_name in dropped_df.columns
]

state_list = ["normal", "abnormal", "recovery"]

dropped_profile_frames = []
dropped_profile_paths = {}

for state_value in state_list:
    dropped_state_frame = dropped_df.loc[
        dropped_df[state_column_synthetic] == state_value
    ].copy()

    dropped_state_profile = build_state_sensor_profile_table(
        dropped_state_frame,
        feature_columns=dropped_present,
        state_value=state_value,
        include_missing_pct=True,
        keep_empty_rows=True,
    )

    if dropped_state_profile.empty:
        continue

    dropped_profile_frames.append(dropped_state_profile)

    state_path = (
        SILVER_EDA_ARTIFACTS_PATH
        / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__{state_value}.csv"
    )
    dropped_state_profile.to_csv(state_path, index=False)
    dropped_profile_paths[f"dropped_feature_profiles_{state_value}_path"] = str(state_path)

if len(dropped_profile_frames) > 0:
    dropped_profiles_df = pd.concat(dropped_profile_frames, axis=0, ignore_index=True)

    combined_path = (
        SILVER_EDA_ARTIFACTS_PATH
        / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__all_states.csv"
    )
    dropped_profiles_df.to_csv(combined_path, index=False)
    dropped_profile_paths["dropped_feature_profiles_all_states_path"] = str(combined_path)

    print("Saved dropped profile paths:", dropped_profile_paths)
    display(dropped_profiles_df.head(20))
else:
    dropped_profiles_df = pd.DataFrame()
    print("No dropped sensor profiles were generated.")

In [ ]:
dropped_profile_paths = {}

combined_path = (
    SILVER_EDA_ARTIFACTS_PATH
    / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__all_states.csv"
)
dropped_profiles_df.to_csv(combined_path, index=False)
dropped_profile_paths["dropped_feature_profiles_all_states_path"] = str(combined_path)

for state_value in ["normal", "abnormal", "recovery"]:
    state_subset = dropped_profiles_df.loc[
        dropped_profiles_df["state_scope"] == state_value
    ].copy()

    state_path = (
        SILVER_EDA_ARTIFACTS_PATH
        / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__{state_value}.csv"
    )
    state_subset.to_csv(state_path, index=False)
    dropped_profile_paths[f"dropped_feature_profiles_{state_value}_path"] = str(state_path)

print("Saved dropped profile paths:")
for key, value in dropped_profile_paths.items():
    print(f"{key}: {value}")

In [ ]:
def build_dropped_sensor_profiles_from_silver_preeda_truth(
    *,
    silver_truth: dict,
    silver_dataframe: pd.DataFrame,
    dataset_name: str,
    artifacts_dir: Path,
    truths_path: Path,
    state_column: str,
    synthetic_state_column: str,
    join_key: str = "meta__record_id",
    logger=None,
) -> dict:
    """
    Load dropped-sensor artifacts from Silver Pre-EDA truth, attach state columns,
    build per-state dropped-feature profiles, and save outputs.
    """
    silver_preeda_truth_hash = get_truth_hash(silver_truth)

    silver_preeda_truth = load_truth_record_by_hash(
        truth_dir=truths_path,
        layer_name="silver",
        dataset_name=dataset_name,
        truth_hash=str(silver_preeda_truth_hash).strip(),
    )

    runtime_facts = silver_preeda_truth.get("runtime_facts", {}) or {}
    artifact_paths = silver_preeda_truth.get("artifact_paths", {}) or {}

    drop_audit = runtime_facts.get("sensor_drop_audit", {}) or {}
    dropped_sensors = drop_audit.get("dropped_features", []) or []
    dropped_parquet_path = artifact_paths.get("silver_preeda_dropped_sensors_parquet")

    has_dropped = bool(dropped_sensors) and bool(dropped_parquet_path)

    if not has_dropped:
        if logger is not None:
            logger.info("No dropped sensors detected; skipped dropped-sensor profiling.")

        return {
            "has_dropped": False,
            "silver_preeda_truth_hash": silver_preeda_truth_hash,
            "silver_preeda_truth": silver_preeda_truth,
            "dropped_sensors": dropped_sensors,
            "dropped_parquet_path": dropped_parquet_path,
            "dropped_dataframe": pd.DataFrame(),
            "dropped_profiles_df": pd.DataFrame(),
            "dropped_profile_paths": {},
        }

    dropped_df = load_dropped_sensor_dataframe(
        dropped_path=dropped_parquet_path,
    )

    dropped_df = attach_state_column_to_dropped_dataframe(
        dropped_dataframe=dropped_df,
        silver_dataframe=silver_dataframe,
        state_column=state_column,
        synthetic_state_column=synthetic_state_column,
        join_key=join_key,
    )

    dropped_present = [
        sensor_name
        for sensor_name in dropped_sensors
        if sensor_name in dropped_df.columns
    ]

    if len(dropped_present) == 0:
        raise ValueError(
            "Dropped parquet contains none of the dropped sensors listed in truth."
        )

    state_list = ["normal", "abnormal", "recovery"]

    dropped_profiles_df = profile_sensor_state_table(
        dropped_df,
        sensors=dropped_present,
        state_values=state_list,
        state_column=synthetic_state_column,
    )

    artifacts_dir.mkdir(parents=True, exist_ok=True)

    dropped_profile_paths = {}

    combined_path = (
        artifacts_dir
        / f"{dataset_name}__silver_eda__dropped_feature_profiles__all_states.csv"
    )
    dropped_profiles_df.to_csv(combined_path, index=False)
    dropped_profile_paths["dropped_feature_profiles_all_states_path"] = str(combined_path)

    for state_value in state_list:
        state_subset = dropped_profiles_df.loc[
            dropped_profiles_df["state_scope"] == state_value
        ].copy()

        state_path = (
            artifacts_dir
            / f"{dataset_name}__silver_eda__dropped_feature_profiles__{state_value}.csv"
        )
        state_subset.to_csv(state_path, index=False)
        dropped_profile_paths[
            f"dropped_feature_profiles_{state_value}_path"
        ] = str(state_path)

    return {
        "has_dropped": True,
        "silver_preeda_truth_hash": silver_preeda_truth_hash,
        "silver_preeda_truth": silver_preeda_truth,
        "dropped_sensors": dropped_sensors,
        "dropped_parquet_path": str(dropped_parquet_path),
        "dropped_dataframe": dropped_df,
        "dropped_profiles_df": dropped_profiles_df,
        "dropped_profile_paths": dropped_profile_paths,
    }

In [ ]:
dropped_sensor_artifacts = build_dropped_sensor_profiles_from_silver_preeda_truth(
    silver_truth=silver_truth,
    silver_dataframe=dataframe,
    dataset_name=DATASET_NAME,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    truths_path=TRUTHS_PATH,
    state_column=state_column,
    synthetic_state_column=state_column_synthetic,
    join_key="meta__record_id",
    logger=logger,
)

HAS_DROPPED = dropped_sensor_artifacts["has_dropped"]
dropped_sensors = dropped_sensor_artifacts["dropped_sensors"]
dropped_parquet_path = dropped_sensor_artifacts["dropped_parquet_path"]
dropped_df = dropped_sensor_artifacts["dropped_dataframe"]
dropped_profiles_df = dropped_sensor_artifacts["dropped_profiles_df"]
dropped_profile_paths = dropped_sensor_artifacts["dropped_profile_paths"]

print("HAS_DROPPED:", HAS_DROPPED)
print("Dropped sensors:", dropped_sensors)
print("Dropped parquet path:", dropped_parquet_path)

if HAS_DROPPED:
    print("Saved dropped profile paths:")
    for key, value in dropped_profile_paths.items():
        print(f"{key}: {value}")

    display(dropped_profiles_df.head(20))
else:
    print("No dropped sensors detected; skipped dropped-sensor profiling.")

### Ask

Why bother profiling dropped sensors if they were already removed from the main feature set?

### Answer

Because feature removal and feature irrelevance are not always the same thing.

A sensor may be dropped for missingness or stability reasons and still show recognizable behavior by state. That can still be useful for documentation, future tuning, or synthetic-data planning.

So this section helps preserve the reasoning around what was dropped instead of making those columns disappear without explanation.

----

## Define the Episode-by-State Summary Logic

This helper function builds a structured view of how operating states are distributed across episodes.

The point of this section is to move beyond row counts and look at the data in a more episode-aware way. That matters because this project is not just about isolated rows. It is also about how states behave within failure and recovery periods over time.

In [ ]:
def get_episode_status_state_stats(
    dataframe: pd.DataFrame,
    status_column: str = "machine_status",
    episode_column: str = "meta__episode_id",
    state_order: list | None = None,
    include_null_episode: bool = False,
    state_map: dict | None = None,
    lowercase_states: bool = True,
    strip_states: bool = True,
    percent_suffix: str = "_percent",
) -> dict:
    """
    Build global and episode-level state summary tables.
    """
    required_columns = [status_column, episode_column]
    missing = [column for column in required_columns if column not in dataframe.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    work_dataframe = dataframe.copy()

    state_series = work_dataframe[status_column].copy()
    null_mask = state_series.isna()

    state_series = state_series.astype("string")

    if strip_states:
        state_series = state_series.str.strip()

    if lowercase_states:
        state_series = state_series.str.lower()

    if state_map is not None:
        state_series = state_series.map(state_map).fillna(state_series)

    state_series = state_series.where(~null_mask, pd.NA)

    normalized_status_column = f"{status_column}_normalized"
    work_dataframe[normalized_status_column] = state_series

    global_status_stats = (
        work_dataframe[normalized_status_column]
        .value_counts(dropna=False)
        .rename_axis("status_state")
        .reset_index(name="global_count")
    )

    total_rows = global_status_stats["global_count"].sum()
    global_status_stats["global_percent"] = (
        global_status_stats["global_count"] / total_rows * 100
    ).round(2)

    if include_null_episode:
        episode_dataframe = work_dataframe.copy()
    else:
        episode_dataframe = work_dataframe[
            work_dataframe[episode_column].notna()
        ].copy()

    episode_status_counts = (
        episode_dataframe.groupby([episode_column, normalized_status_column], dropna=False)
        .size()
        .unstack(fill_value=0)
    )

    if state_order is None:
        state_order = list(episode_status_counts.columns)

    for state in state_order:
        if state not in episode_status_counts.columns:
            episode_status_counts[state] = 0

    episode_status_counts = episode_status_counts[state_order]
    episode_status_counts = episode_status_counts.reset_index()

    status_columns = [column for column in episode_status_counts.columns if column != episode_column]

    episode_status_counts["episode_total_rows"] = episode_status_counts[status_columns].sum(axis=1)

    for state in status_columns:
        episode_status_counts[f"{state}{percent_suffix}"] = (
            episode_status_counts[state] / episode_status_counts["episode_total_rows"]
        )

    reordered_columns = [episode_column]
    for state in status_columns:
        reordered_columns.append(state)
        reordered_columns.append(f"{state}{percent_suffix}")
    reordered_columns.append("episode_total_rows")

    episode_status_counts = episode_status_counts[reordered_columns]
    episode_status_counts.columns.name = None
    episode_status_counts.index.name = None

    episode_status_means = pd.DataFrame(
        {
            "status_state": status_columns,
            "mean_rows_per_episode": [
                round(float(episode_status_counts[state].mean()), 2)
                for state in status_columns
            ],
        }
    )

    episode_status_percent_means = pd.DataFrame(
        {
            "status_state": status_columns,
            "mean_percent_per_episode": [
                float(episode_status_counts[f"{state}{percent_suffix}"].mean())
                for state in status_columns
            ],
        }
    )

    mean_total_rows = (
        round(episode_status_counts["episode_total_rows"].mean(), 2)
        if len(episode_status_counts)
        else 0.0
    )

    episode_totals = pd.DataFrame(
        {
            "metric": [
                "episode_count",
                "mean_total_rows_per_episode",
            ],
            "value": [
                int(episode_status_counts[episode_column].nunique()) if len(episode_status_counts) else 0,
                mean_total_rows,
            ],
        }
    )

    return {
        "global_status_stats": global_status_stats,
        "episode_status_counts": episode_status_counts,
        "episode_status_means": episode_status_means,
        "episode_status_percent_means": episode_status_percent_means,
        "episode_totals": episode_totals,
        "normalized_status_column": normalized_status_column,
        "status_columns": status_columns,
        "percent_suffix": percent_suffix,
    }

In [ ]:
EPISODE_COLUMN = "meta__episode_id"

if EPISODE_COLUMN not in dataframe.columns:
    raise KeyError(f"Required episode column '{EPISODE_COLUMN}' not found in dataframe.")

if state_column_synthetic not in dataframe.columns:
    raise KeyError(
        f"Normalized state column '{state_column_synthetic}' not found in dataframe."
    )

episode_status_stats = get_episode_status_state_stats(
    dataframe=dataframe,
    status_column=state_column_synthetic,
    episode_column=EPISODE_COLUMN,
    state_order=["normal", "abnormal", "recovery"],
    include_null_episode=False,
    state_map=None,          # already normalized earlier
    lowercase_states=False,  # already normalized earlier
    strip_states=False,      # already normalized earlier
)

global_status_stats = episode_status_stats["global_status_stats"]
episode_status_counts = episode_status_stats["episode_status_counts"]
episode_status_means = episode_status_stats["episode_status_means"]
episode_status_percent_means = episode_status_stats["episode_status_percent_means"]
episode_totals = episode_status_stats["episode_totals"]

display(global_status_stats)
display(episode_status_counts)
display(episode_status_means)
display(episode_status_percent_means)
display(episode_totals)

episode_status_counts_display = episode_status_counts.copy()
percent_columns = [
    column for column in episode_status_counts_display.columns
    if column.endswith("_percent")
]
episode_status_counts_display[percent_columns] = (
    episode_status_counts_display[percent_columns].round(6)
)

display(episode_status_counts_display)

In [ ]:
episode_totals_lookup = (
    dict(zip(episode_totals["metric"], episode_totals["value"]))
    if isinstance(episode_totals, pd.DataFrame) and not episode_totals.empty
    else {}
)

episode_status_mean_lookup = (
    dict(zip(episode_status_means["status_state"], episode_status_means["mean_rows_per_episode"]))
    if isinstance(episode_status_means, pd.DataFrame) and not episode_status_means.empty
    else {}
)

episode_status_percent_mean_lookup = (
    dict(zip(episode_status_percent_means["status_state"], episode_status_percent_means["mean_percent_per_episode"]))
    if isinstance(episode_status_percent_means, pd.DataFrame) and not episode_status_percent_means.empty
    else {}
)

global_status_count_lookup = (
    dict(zip(global_status_stats["status_state"], global_status_stats["global_count"]))
    if isinstance(global_status_stats, pd.DataFrame) and not global_status_stats.empty
    else {}
)

global_status_percent_lookup = (
    dict(zip(global_status_stats["status_state"], global_status_stats["global_percent"]))
    if isinstance(global_status_stats, pd.DataFrame) and not global_status_stats.empty
    else {}
)

episode_status_counts_path = (
    SILVER_EDA_ARTIFACTS_PATH / f"{DATASET_NAME}__silver_eda__episode_status_counts.json"
)

episode_status_counts_records = (
    episode_status_counts
    .where(pd.notna(episode_status_counts), None)
    .to_dict(orient="records")
)

with open(episode_status_counts_path, "w", encoding="utf-8") as file:
    json.dump(episode_status_counts_records, file, indent=2)

print("Episode status counts path:", episode_status_counts_path)

EPISODE_STATUS_STATE_STATS_PAYLOAD = {
    "status_column": str(state_column_synthetic),
    "episode_column": EPISODE_COLUMN,
    "episode_count": int(episode_totals_lookup.get("episode_count", 0) or 0),
    "mean_total_rows_per_episode": float(
        episode_totals_lookup.get("mean_total_rows_per_episode", 0.0) or 0.0
    ),
    "global_status_stats": (
        global_status_stats.where(pd.notna(global_status_stats), None).to_dict(orient="records")
        if isinstance(global_status_stats, pd.DataFrame) and not global_status_stats.empty
        else []
    ),
    "episode_status_means": (
        episode_status_means.where(pd.notna(episode_status_means), None).to_dict(orient="records")
        if isinstance(episode_status_means, pd.DataFrame) and not episode_status_means.empty
        else []
    ),
    "episode_status_percent_means": (
        episode_status_percent_means.where(pd.notna(episode_status_percent_means), None).to_dict(orient="records")
        if isinstance(episode_status_percent_means, pd.DataFrame) and not episode_status_percent_means.empty
        else []
    ),
    "episode_totals": (
        episode_totals.where(pd.notna(episode_totals), None).to_dict(orient="records")
        if isinstance(episode_totals, pd.DataFrame) and not episode_totals.empty
        else []
    ),
    "episode_status_mean_lookup": {
        str(key): float(value) for key, value in episode_status_mean_lookup.items()
    },
    "episode_status_percent_mean_lookup": {
        str(key): float(value) for key, value in episode_status_percent_mean_lookup.items()
    },
    "global_status_count_lookup": {
        str(key): int(value) for key, value in global_status_count_lookup.items()
    },
    "global_status_percent_lookup": {
        str(key): float(value) for key, value in global_status_percent_lookup.items()
    },
}



ledger.add(
    kind="step",
    step="episode_status_state_stats",
    message="Computed global and per-episode normalized state statistics from Silver episode IDs.",
    data={
        "status_column": str(state_column_synthetic),
        "episode_column": EPISODE_COLUMN,
        "episode_count": int(EPISODE_STATUS_STATE_STATS_PAYLOAD["episode_count"]),
        "mean_total_rows_per_episode": float(
            EPISODE_STATUS_STATE_STATS_PAYLOAD["mean_total_rows_per_episode"]
        ),
        "mean_rows_normal": float(episode_status_mean_lookup.get("normal", 0.0) or 0.0),
        "mean_rows_abnormal": float(episode_status_mean_lookup.get("abnormal", 0.0) or 0.0),
        "mean_rows_recovery": float(episode_status_mean_lookup.get("recovery", 0.0) or 0.0),
        "mean_percent_normal": float(episode_status_percent_mean_lookup.get("normal", 0.0) or 0.0),
        "mean_percent_abnormal": float(episode_status_percent_mean_lookup.get("abnormal", 0.0) or 0.0),
        "mean_percent_recovery": float(episode_status_percent_mean_lookup.get("recovery", 0.0) or 0.0),
        "global_count_normal": int(global_status_count_lookup.get("normal", 0) or 0),
        "global_count_abnormal": int(global_status_count_lookup.get("abnormal", 0) or 0),
        "global_count_recovery": int(global_status_count_lookup.get("recovery", 0) or 0),
        "global_percent_normal": float(global_status_percent_lookup.get("normal", 0.0) or 0.0),
        "global_percent_abnormal": float(global_status_percent_lookup.get("abnormal", 0.0) or 0.0),
        "global_percent_recovery": float(global_status_percent_lookup.get("recovery", 0.0) or 0.0),
        "episode_status_counts_path": str(episode_status_counts_path),
    },
    logger=logger,
)

### Ask

Why am I looking at state behavior by episode instead of only by total row count?

### Answer

Because row totals alone can hide how the process actually behaves over time.

Two datasets can have the same number of abnormal rows but very different episode patterns. One could have many short disruptions, while another could have fewer but longer broken periods. That difference matters for both modeling and synthetic-data design.

So this section is really helping answer: **what does the state structure look like across actual operational episodes, not just across all rows mixed together?**

---

## Build normal-state correlation relationships

To understand baseline sensor relationships, the notebook computes pairwise correlations using only rows from the normalized `normal` state.

This focuses the analysis on expected system behavior rather than mixing in fault or recovery conditions, which could distort the relationships between sensors.

The output includes both:

- a full correlation matrix for plotting and grouping
- a long-form sensor pair table for downstream filtering and artifact export

In [ ]:
def build_normal_only_correlation_pairs(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    state_column: str,
    target_state: str = "normal",
) -> dict:
    """
    Build normal-only correlation matrix and pair table.
    """

    if state_column not in dataframe.columns:
        raise KeyError(f"Missing state column: {state_column}")

    use_features = [
        column for column in feature_columns
        if column in dataframe.columns
    ]

    if len(use_features) == 0:
        return {
            "correlation_matrix": pd.DataFrame(),
            "correlation_pairs": pd.DataFrame(),
        }

    normal_df = dataframe.loc[
        dataframe[state_column].eq(target_state),
        use_features,
    ].copy()

    for column in normal_df.columns:
        normal_df[column] = pd.to_numeric(normal_df[column], errors="coerce")

    normal_df = normal_df.dropna(axis=1, how="all")

    if normal_df.empty or normal_df.shape[1] == 0:
        return {
            "correlation_matrix": pd.DataFrame(),
            "correlation_pairs": pd.DataFrame(),
        }

    correlation_matrix = normal_df.corr(numeric_only=True)

    pair_rows = []
    for sensor_a in correlation_matrix.columns:
        for sensor_b in correlation_matrix.columns:
            if sensor_a >= sensor_b:
                continue

            corr_value = correlation_matrix.loc[sensor_a, sensor_b]

            pair_rows.append(
                {
                    "sensor_a": str(sensor_a),
                    "sensor_b": str(sensor_b),
                    "correlation": float(corr_value) if pd.notna(corr_value) else None,
                    "abs_correlation": abs(float(corr_value)) if pd.notna(corr_value) else None,
                }
            )

    correlation_pairs = (
        pd.DataFrame(pair_rows)
        .sort_values(
            ["abs_correlation", "sensor_a", "sensor_b"],
            ascending=[False, True, True],
        )
        .reset_index(drop=True)
        if pair_rows
        else pd.DataFrame()
    )

    return {
        "correlation_matrix": correlation_matrix,
        "correlation_pairs": correlation_pairs,
    }

In [ ]:
correlation_bundle = build_normal_only_correlation_pairs(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    state_column=state_column_synthetic,
    target_state="normal",
)

correlation_matrix_normal = correlation_bundle["correlation_matrix"]
sensor_correlation_pairs_normal_df = correlation_bundle["correlation_pairs"]

correlation_matrix_normal_path = (
    SILVER_EDA_ARTIFACTS_PATH / "correlation_matrix_normal.csv"
)
sensor_correlation_pairs_normal_path = (
    SILVER_EDA_ARTIFACTS_PATH / "sensor_correlation_pairs_normal.csv"
)

save_data(
    correlation_matrix_normal,
    SILVER_EDA_ARTIFACTS_PATH,
    correlation_matrix_normal_path.name,
)

save_data(
    sensor_correlation_pairs_normal_df,
    SILVER_EDA_ARTIFACTS_PATH,
    sensor_correlation_pairs_normal_path.name,
)

print("Saved:", correlation_matrix_normal_path)
print("Saved:", sensor_correlation_pairs_normal_path)

display(sensor_correlation_pairs_normal_df.head(20))

----


## Group sensors into correlated baseline subsystems

The correlation matrix is next converted into connected sensor groups.

These groups do not claim true physical subsystem membership, but they provide a practical approximation of which sensors move together strongly under normal behavior. This is useful for downstream synthetic fault propagation, where faults may spread more readily between tightly related sensors.

In [ ]:
def find(parent: dict, x: str) -> str:
    """
    Disjoint-set find with path compression.
    """
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]


def union(parent: dict, a: str, b: str) -> None:
    """
    Disjoint-set union.
    """
    root_a = find(parent, a)
    root_b = find(parent, b)
    if root_a != root_b:
        parent[root_b] = root_a

In [ ]:
def build_sensor_group_map_from_correlation(
    correlation_matrix: pd.DataFrame,
    *,
    min_abs_corr_for_group: float = 0.60,
) -> pd.DataFrame:
    """
    Build connected-component sensor groups from correlation matrix.
    """
    if correlation_matrix.empty:
        return pd.DataFrame()

    abs_corr = correlation_matrix.abs()
    parent = {}

    for sensor_name in abs_corr.columns:
        find(parent, str(sensor_name))

    for sensor_a in abs_corr.columns:
        for sensor_b in abs_corr.columns:
            if sensor_a >= sensor_b:
                continue

            corr_strength = abs_corr.loc[sensor_a, sensor_b]
            if pd.notna(corr_strength) and float(corr_strength) >= float(min_abs_corr_for_group):
                union(parent, str(sensor_a), str(sensor_b))

    groups = {}
    for sensor_name in abs_corr.columns:
        root = find(parent, str(sensor_name))
        groups.setdefault(root, []).append(str(sensor_name))

    rows = []
    for group_priority, members in enumerate(
        sorted(groups.values(), key=lambda values: (-len(values), values[0])),
        start=1,
    ):
        group_name = f"group_{group_priority:03d}"
        members = sorted(members)

        for position, sensor_name in enumerate(members):
            rows.append(
                {
                    "group_name": group_name,
                    "sensor": sensor_name,
                    "group_size": len(members),
                    "group_priority": group_priority,
                    "group_role": "leader" if position == 0 else "member",
                    "group_method": f"corr_components_abs>={min_abs_corr_for_group}",
                }
            )

    return (
        pd.DataFrame(rows)
        .sort_values(["group_priority", "sensor"])
        .reset_index(drop=True)
        if rows
        else pd.DataFrame()
    )

In [ ]:
MIN_ABS_CORR_FOR_GROUP = 0.60

sensor_group_map_normal_df = build_sensor_group_map_from_correlation(
    correlation_matrix_normal,
    min_abs_corr_for_group=MIN_ABS_CORR_FOR_GROUP,
)

sensor_group_map_normal_path = (
    SILVER_EDA_ARTIFACTS_PATH / "sensor_group_map_normal.csv"
)

save_data(
    sensor_group_map_normal_df,
    SILVER_EDA_ARTIFACTS_PATH,
    sensor_group_map_normal_path.name,
)

print("Saved:", sensor_group_map_normal_path)
display(sensor_group_map_normal_df.head(25))

if not sensor_group_map_normal_df.empty:
    print("Groups:", int(sensor_group_map_normal_df["group_name"].nunique()))
else:
    print("Groups: 0")

----

## Convert strong baseline relationships into candidate fault propagation pairings

The final step in this section converts strong normal-state relationships into directional propagation pairings.

These pairings are not intended to represent confirmed causal physics. Instead, they provide a practical downstream artifact that identifies which sensors are strong candidates to co-fail, react together, or receive delayed secondary effects during synthetic anomaly generation.

Evidence for a pairing is based on:

- correlation strength under normal behavior
- whether both sensors belong to the same correlated sensor group
- a simple lag heuristic that increases as evidence weakens

In [ ]:
def build_fault_propagation_pairings_from_strong_relationships(
    normal_sensor_group_map_dataframe: pd.DataFrame,
    *,
    normal_sensor_correlation_pairs_dataframe: pd.DataFrame,
    min_abs_corr_for_pair: float = 0.65,
    intra_group_bonus: float = 0.15,
) -> pd.DataFrame:
    """
    Build directional fault propagation pairings from strong correlation pairs
    and optional group membership.
    """
    expected_pair_columns = {"sensor_a", "sensor_b", "abs_correlation"}
    missing_pair_columns = expected_pair_columns - set(normal_sensor_correlation_pairs_dataframe.columns)
    if missing_pair_columns:
        raise KeyError(
            f"Correlation pairs dataframe missing required columns: {sorted(missing_pair_columns)}"
        )

    if normal_sensor_correlation_pairs_dataframe.empty:
        return pd.DataFrame(
            columns=[
                "sensor_primary",
                "sensor_secondary",
                "relationship_type",
                "correlation_strength",
                "fault_coupling_strength",
                "recommended_secondary_fault",
                "lag_cycles",
                "same_group_flag",
                "evidence_score",
                "group_name_primary",
                "group_name_secondary",
            ]
        )

    if (
        not normal_sensor_group_map_dataframe.empty
        and {"sensor", "group_name"}.issubset(normal_sensor_group_map_dataframe.columns)
    ):
        sensor_to_group = dict(
            zip(
                normal_sensor_group_map_dataframe["sensor"],
                normal_sensor_group_map_dataframe["group_name"],
            )
        )
    else:
        sensor_to_group = {}

    pairs = normal_sensor_correlation_pairs_dataframe.copy()
    pairs["strength"] = pd.to_numeric(pairs["abs_correlation"], errors="coerce")
    pairs = pairs.loc[pairs["strength"].ge(min_abs_corr_for_pair)].copy()

    if pairs.empty:
        return pd.DataFrame(
            columns=[
                "sensor_primary",
                "sensor_secondary",
                "relationship_type",
                "correlation_strength",
                "fault_coupling_strength",
                "recommended_secondary_fault",
                "lag_cycles",
                "same_group_flag",
                "evidence_score",
                "group_name_primary",
                "group_name_secondary",
            ]
        )

    pair_rows = []

    for _, row in pairs.iterrows():
        sensor_a = str(row["sensor_a"])
        sensor_b = str(row["sensor_b"])
        strength = float(row["strength"]) if pd.notna(row["strength"]) else 0.0

        group_a = sensor_to_group.get(sensor_a)
        group_b = sensor_to_group.get(sensor_b)
        same_group = (group_a is not None) and (group_a == group_b)

        evidence = min(1.0, strength + (intra_group_bonus if same_group else 0.0))

        if evidence >= 0.85:
            lag_cycles = 0
        elif evidence >= 0.70:
            lag_cycles = 1
        elif evidence >= 0.55:
            lag_cycles = 2
        else:
            lag_cycles = 3

        relationship_type = (
            "same_subsystem_coupled"
            if same_group
            else "correlated_coupled"
        )

        for primary, secondary in ((sensor_a, sensor_b), (sensor_b, sensor_a)):
            pair_rows.append(
                {
                    "sensor_primary": primary,
                    "sensor_secondary": secondary,
                    "relationship_type": relationship_type,
                    "correlation_strength": strength,
                    "fault_coupling_strength": evidence,
                    "recommended_secondary_fault": "variance_burst",
                    "lag_cycles": lag_cycles,
                    "same_group_flag": same_group,
                    "evidence_score": evidence,
                    "group_name_primary": sensor_to_group.get(primary),
                    "group_name_secondary": sensor_to_group.get(secondary),
                }
            )

    return (
        pd.DataFrame(pair_rows)
        .sort_values(
            [
                "evidence_score",
                "fault_coupling_strength",
                "correlation_strength",
                "sensor_primary",
                "sensor_secondary",
            ],
            ascending=[False, False, False, True, True],
        )
        .reset_index(drop=True)
    )

In [ ]:
MIN_ABS_CORR_FOR_PAIR = 0.65

sensor_fault_pairings_normal_df = (
    build_fault_propagation_pairings_from_strong_relationships(
        sensor_group_map_normal_df,
        normal_sensor_correlation_pairs_dataframe=sensor_correlation_pairs_normal_df,
        min_abs_corr_for_pair=MIN_ABS_CORR_FOR_PAIR,
    )
)

sensor_fault_pairings_normal_path = (
    SILVER_EDA_ARTIFACTS_PATH / "sensor_fault_pairings_normal.csv"
)

save_data(
    sensor_fault_pairings_normal_df,
    SILVER_EDA_ARTIFACTS_PATH,
    sensor_fault_pairings_normal_path.name,
)

print("Saved:", sensor_fault_pairings_normal_path)
display(sensor_fault_pairings_normal_df.head(25))

----

## Compare Normal and Abnormal Feature Behavior with Effect Size

Here I compare the normal and abnormal feature profiles using a simple effect-size-style measure.

This helps answer a more targeted EDA question:

**which features move the most between normal and abnormal behavior, and in what direction?**

That gives me a ranked list of features that look most behaviorally different during abnormal operation.

In [ ]:
def build_feature_behavior_effect_size_table(
    dataframe: pd.DataFrame,
    *,
    sensors: Sequence[str],
    state_column: str,
    baseline_state: str = "normal",
    comparison_states: Sequence[str] = ("abnormal", "recovery"),
) -> pd.DataFrame:
    """
    Build effect-size-style summary table using standardized mean shift vs baseline state.
    """
    rows = []

    if state_column not in dataframe.columns:
        raise KeyError(f"Missing state column: {state_column}")

    for sensor in sensors:
        if sensor not in dataframe.columns:
            continue

        sensor_series = pd.to_numeric(dataframe[sensor], errors="coerce")

        baseline_values = sensor_series.loc[dataframe[state_column] == baseline_state]
        baseline_mean = float(baseline_values.mean()) if baseline_values.notna().any() else None
        baseline_std = float(baseline_values.std()) if baseline_values.notna().any() else None

        for comparison_state in comparison_states:
            comparison_values = sensor_series.loc[dataframe[state_column] == comparison_state]
            comparison_mean = float(comparison_values.mean()) if comparison_values.notna().any() else None

            standardized_mean_shift = None
            if (
                baseline_mean is not None
                and comparison_mean is not None
                and baseline_std is not None
                and baseline_std != 0
                and not np.isnan(baseline_std)
            ):
                standardized_mean_shift = (comparison_mean - baseline_mean) / baseline_std

            rows.append(
                {
                    "sensor": sensor,
                    "baseline_state": baseline_state,
                    "comparison_state": comparison_state,
                    "baseline_mean": baseline_mean,
                    "comparison_mean": comparison_mean,
                    "baseline_std": baseline_std,
                    "standardized_mean_shift": standardized_mean_shift,
                }
            )

    return pd.DataFrame(rows)

In [ ]:
effect_table = build_feature_behavior_effect_size_table(
    dataframe,
    sensors=FEATURE_COLUMNS,
    state_column=state_column_synthetic,
    baseline_state="normal",
    comparison_states=("abnormal", "recovery"),
)

if effect_table.empty:
    logger.info("Effect-size table is empty.")
    TOP_FEATURES = []
else:
    effect_table["abs_standardized_mean_shift"] = (
        effect_table["standardized_mean_shift"].abs()
    )

    effect_table = effect_table.sort_values(
        ["comparison_state", "abs_standardized_mean_shift", "sensor"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

effect_path = SILVER_EDA_ARTIFACTS_PATH / "feature_behavior__effect_size.csv"
effect_table.to_csv(effect_path, index=False)
wandb.save(str(effect_path))

# Use abnormal-vs-normal shifts to choose top features for plotting
top_feature_source = effect_table.loc[
    effect_table["comparison_state"].eq("abnormal")
].copy()

if effect_table.empty or "comparison_state" not in effect_table.columns:
    TOP_FEATURES = []
else:
    top_feature_source = effect_table.loc[
        effect_table["comparison_state"].eq("abnormal")
    ].copy()

    if top_feature_source.empty:
        top_feature_source = effect_table.copy()

    TOP_FEATURES = (
        top_feature_source["sensor"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(25)
        .tolist()
    )

logger.info("Selected %d TOP_FEATURES from effect-size table.", len(TOP_FEATURES))
display(effect_table.head(25))
print("TOP_FEATURES:", TOP_FEATURES)

### Ask

What does the effect-size table help me understand?

### Answer

It helps me move from “this feature looks different” to “how different does it look relative to its normal variation?”

That matters because a raw mean shift alone can be misleading. A feature that changes by a certain amount may be very important if it is usually stable, but less important if it already varies a lot under normal conditions.

So this table helps rank features by how strongly they separate normal and abnormal behavior in a more comparable way.

----

In [ ]:
def build_plot_feature_list(
    dataframe: pd.DataFrame,
    *,
    feature_candidates: Sequence[str],
    max_features: Optional[int] = None,
) -> list[str]:
    """
    Keep only numeric features that actually exist in the dataframe.
    """
    selected = []
    seen = set()

    for feature_name in feature_candidates:
        feature_name = str(feature_name)
        if feature_name in seen:
            continue
        if feature_name not in dataframe.columns:
            continue
        if not pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            continue

        selected.append(feature_name)
        seen.add(feature_name)

        if max_features is not None and len(selected) >= int(max_features):
            break

    return selected


In [ ]:

def build_state_plot_frames(
    dataframe: pd.DataFrame,
    *,
    state_column: str,
    feature_columns: Sequence[str],
    state_values: Sequence[str] = ("normal", "abnormal", "recovery"),
) -> dict[str, pd.DataFrame]:
    """
    Build numeric per-state plotting frames.
    """
    if state_column not in dataframe.columns:
        raise KeyError(f"Missing state column: {state_column}")

    frames = {}

    for state_value in state_values:
        state_frame = dataframe.loc[
            dataframe[state_column].eq(state_value),
            [column for column in feature_columns if column in dataframe.columns],
        ].copy()

        for column_name in state_frame.columns:
            state_frame[column_name] = pd.to_numeric(
                state_frame[column_name],
                errors="coerce",
            )

        state_frame = state_frame.dropna(axis=1, how="all")
        frames[str(state_value)] = state_frame

    return frames



In [ ]:
plot_features = build_plot_feature_list(
    dataframe,
    feature_candidates=(TOP_FEATURES if len(TOP_FEATURES) > 0 else FEATURE_COLUMNS),
    max_features=25,
)

state_plot_frames = build_state_plot_frames(
    dataframe,
    state_column=state_column_synthetic,
    feature_columns=plot_features,
    state_values=("normal", "abnormal", "recovery"),
)

normal_dataframe = state_plot_frames.get("normal", pd.DataFrame())
abnormal_dataframe = state_plot_frames.get("abnormal", pd.DataFrame())
recovery_dataframe = state_plot_frames.get("recovery", pd.DataFrame())

print("plot_features:", plot_features)
print("normal_dataframe:", normal_dataframe.shape)
print("abnormal_dataframe:", abnormal_dataframe.shape)
print("recovery_dataframe:", recovery_dataframe.shape)

---

## Create a Correlation Heatmap for the Top Features

After identifying the strongest behavior-shift features, I use the top subset to build a cleaner correlation heatmap.

This gives me a more focused view of how the highest-interest features relate to each other under normal conditions. It is a simpler visual summary than reviewing the full correlation table by itself.

In [ ]:
def plot_correlation_heatmap(
    correlation_matrix: pd.DataFrame,
    *,
    feature_names: Sequence[str],
    out_path: Path,
    title: str,
) -> Optional[Path]:
    """
    Plot a correlation heatmap for the selected features.
    """
    selected_features = [
        feature_name
        for feature_name in feature_names
        if feature_name in correlation_matrix.index and feature_name in correlation_matrix.columns
    ]

    if len(selected_features) == 0:
        return None

    plot_matrix = correlation_matrix.loc[selected_features, selected_features].copy()

    plt.figure(figsize=(10, 8))
    plt.imshow(plot_matrix.values, aspect="auto")
    plt.title(title)
    plt.colorbar()
    plt.xticks(range(len(selected_features)), selected_features, rotation=90, fontsize=7)
    plt.yticks(range(len(selected_features)), selected_features, fontsize=7)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()
    plt.close()

    return out_path



In [ ]:
correlation_heatmap_path = plot_correlation_heatmap(
    correlation_matrix_normal,
    feature_names=plot_features,
    out_path=SILVER_EDA_ARTIFACTS_PATH / "correlation__normal_heatmap.png",
    title="Correlation heatmap (NORMAL) — selected features",
)

if correlation_heatmap_path is not None:
    wandb.save(str(correlation_heatmap_path))

display(
    correlation_matrix_normal.loc[
        [feature for feature in plot_features if feature in correlation_matrix_normal.index],
        [feature for feature in plot_features if feature in correlation_matrix_normal.columns],
    ].round(3)
)

----

## Compare Normal and Abnormal Feature Distributions

This section creates distribution plots for the top features using normal and abnormal slices.

The goal here is to visually inspect whether the most important features from the effect-size table actually show meaningful separation in their value distributions. Sometimes the table suggests a difference, and the plots help confirm whether that difference looks broad, narrow, shifted, or heavily overlapping.

In [ ]:
def plot_state_distribution_histograms(
    state_frames: dict[str, pd.DataFrame],
    *,
    features: Sequence[str],
    artifacts_dir: Path,
    baseline_state: str = "normal",
    comparison_state: str = "abnormal",
    bins: int = 50,
) -> list[str]:
    """
    Plot baseline vs comparison histograms for each feature.
    """
    saved_paths = []

    baseline_frame = state_frames.get(baseline_state, pd.DataFrame())
    comparison_frame = state_frames.get(comparison_state, pd.DataFrame())

    for feature_name in features:
        if feature_name not in baseline_frame.columns:
            continue
        if feature_name not in comparison_frame.columns:
            continue

        series_baseline = pd.to_numeric(baseline_frame[feature_name], errors="coerce").dropna()
        series_comparison = pd.to_numeric(comparison_frame[feature_name], errors="coerce").dropna()

        if series_baseline.empty and series_comparison.empty:
            continue

        plt.figure(figsize=(10, 4))
        plt.hist(series_baseline.values, bins=bins, alpha=0.5, label=baseline_state.title())
        plt.hist(series_comparison.values, bins=bins, alpha=0.5, label=comparison_state.title())
        plt.title(f"Distribution: {feature_name} ({baseline_state} vs {comparison_state})")
        plt.legend()
        plt.tight_layout()

        out_path = artifacts_dir / f"distribution__{feature_name}.png"
        plt.savefig(out_path, dpi=200)
        plt.show()
        plt.close()

        saved_paths.append(str(out_path))

    return saved_paths



In [ ]:
distribution_plot_paths = plot_state_distribution_histograms(
    state_plot_frames,
    features=plot_features,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    baseline_state="normal",
    comparison_state="abnormal",
    bins=50,
)

for path_value in distribution_plot_paths:
    wandb.save(path_value)

print("Distribution plots saved:", len(distribution_plot_paths))

### Ask

What should I look for in these normal vs abnormal distribution plots?

### Answer

I mainly want to see whether the two states overlap heavily or whether they separate in a way that could actually help anomaly detection.

A few useful patterns are:
- a clear shift in center
- a wider abnormal spread
- a one-sided tail increase
- a tighter normal range with a noisier abnormal range

These plots do not prove that one feature should be modeled by itself, but they do help show which features seem most behaviorally informative.

---

## Helper Functions

In [ ]:
def build_flag_spans(
    dataframe: pd.DataFrame,
    *,
    flag_column: str,
    time_column: str,
) -> list[tuple[float, float]]:
    """
    Build contiguous spans where a binary flag is active.
    """
    if flag_column not in dataframe.columns:
        return []

    working = dataframe[[time_column, flag_column]].copy()
    working[time_column] = pd.to_numeric(working[time_column], errors="coerce")
    working[flag_column] = pd.to_numeric(working[flag_column], errors="coerce").fillna(0).astype(int)
    working = working.dropna(subset=[time_column]).sort_values(time_column).reset_index(drop=True)

    if working.empty:
        return []

    flag_values = working[flag_column].to_numpy()
    time_values = working[time_column].to_numpy()

    spans = []
    start_value = None

    for idx, flag_value in enumerate(flag_values):
        if flag_value == 1 and start_value is None:
            start_value = float(time_values[idx])

        is_last = idx == len(flag_values) - 1
        next_flag = flag_values[idx + 1] if not is_last else 0

        if start_value is not None and (flag_value == 1 and next_flag == 0):
            end_value = float(time_values[idx])
            spans.append((start_value, end_value))
            start_value = None

    return spans


In [ ]:
def resolve_time_axis_series(
    dataframe: pd.DataFrame,
    *,
    preferred_columns: Sequence[str] = ("event_step", "time_index"),
) -> tuple[str, pd.Series]:
    """
    Resolve a time-like axis for plotting.
    """
    for column_name in preferred_columns:
        if column_name in dataframe.columns:
            return column_name, pd.to_numeric(dataframe[column_name], errors="coerce")

    fallback_series = pd.Series(np.arange(len(dataframe), dtype=np.int64), index=dataframe.index)
    return "row_index", fallback_series


----

## Overlay the Top Features Across Time

This plot puts the top features onto one shared z-scored time-based view.

I use this to see whether the strongest behavior-shift features also show visible movement around the anomaly periods when viewed across the sequence of observations. Since the features are z-scored, the point is comparison of movement shape rather than comparison of raw units.

In [ ]:
def plot_top_feature_overlay(
    dataframe: pd.DataFrame,
    *,
    features: Sequence[str],
    artifacts_dir: Path,
    anomaly_flag_column: str = "anomaly_flag",
    normalize: bool = True,
) -> Optional[str]:
    """
    Plot multiple top features on one shared time axis.
    """
    if len(features) == 0:
        return None

    time_column, x_axis = resolve_time_axis_series(dataframe)

    plt.figure(figsize=(12, 6))

    for feature_name in features:
        if feature_name not in dataframe.columns:
            continue
        if not pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            continue

        series = pd.to_numeric(dataframe[feature_name], errors="coerce")
        median_value = float(series.median()) if series.notna().any() else 0.0
        series = series.fillna(median_value)

        plot_series = z_score(series) if normalize else series
        plt.plot(x_axis, plot_series, label=feature_name)

    if anomaly_flag_column in dataframe.columns:
        anomaly_positions = dataframe.index[
            pd.to_numeric(dataframe[anomaly_flag_column], errors="coerce").fillna(0).astype(int).eq(1)
        ].tolist()

        if len(anomaly_positions) > 0:
            step = max(1, len(anomaly_positions) // 200)
            marker_positions = anomaly_positions[::step]
            x_markers = x_axis.iloc[marker_positions] if hasattr(x_axis, "iloc") else np.asarray(x_axis)[marker_positions]
            y_markers = np.zeros(len(marker_positions))
            plt.scatter(x_markers, y_markers, marker="x", label="anomaly_flag=1")

    plt.title("Top feature overlay (z-scored)" if normalize else "Top feature overlay")
    plt.legend(fontsize=7)
    plt.tight_layout()

    out_path = artifacts_dir / "timeseries__top_features_overlay.png"
    plt.savefig(out_path, dpi=200)
    plt.show()
    plt.close()

    return str(out_path)


In [ ]:
overlay_path = plot_top_feature_overlay(
    dataframe,
    features=plot_features,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    anomaly_flag_column="anomaly_flag",
    normalize=True,
)

if overlay_path is not None:
    wandb.save(overlay_path)

print("Overlay path:", overlay_path)

---

## Plot Each Top Feature Against Broken or Anomalous Periods

This section looks at the top features one at a time across the timeline and marks where broken or anomalous periods occur.

I like this view because it is easier to read than the shared overlay when I want to see how a single feature behaves around the abnormal periods. It helps show whether the feature spikes, drifts, drops, or stays noisy near the failure windows.

In [ ]:

def plot_feature_timeseries_with_flag_spans(
    dataframe: pd.DataFrame,
    *,
    features: Sequence[str],
    artifacts_dir: Path,
    flag_column: str = "anomaly_flag",
    normalize: bool = True,
) -> list[str]:
    """
    Plot one time-series chart per feature with anomaly spans highlighted.
    """
    if len(features) == 0:
        return []

    time_column, x_axis = resolve_time_axis_series(dataframe)
    flag_spans = build_flag_spans(
        dataframe,
        flag_column=flag_column,
        time_column=time_column,
    ) if flag_column in dataframe.columns else []

    saved_paths = []

    for feature_name in features:
        if feature_name not in dataframe.columns:
            continue
        if not pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            continue

        series = pd.to_numeric(dataframe[feature_name], errors="coerce")
        median_value = float(series.median()) if series.notna().any() else 0.0
        series = series.fillna(median_value)

        plot_series = z_score(series) if normalize else series

        plt.figure(figsize=(18, 3))
        plt.plot(x_axis, plot_series, label=feature_name)

        for start_value, end_value in flag_spans:
            plt.axvspan(start_value, end_value, alpha=0.12)

        plt.title(f"Time series - {feature_name}")
        plt.legend(fontsize=7)
        plt.tight_layout()

        out_path = artifacts_dir / f"timeseries__{feature_name}__flagged.png"
        plt.savefig(out_path, dpi=200)
        plt.show()
        plt.close()

        saved_paths.append(str(out_path))

    return saved_paths



In [ ]:
feature_timeseries_plot_paths = plot_feature_timeseries_with_flag_spans(
    dataframe,
    features=plot_features,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    flag_column="anomaly_flag",
    normalize=True,
)

for path_value in feature_timeseries_plot_paths:
    wandb.save(path_value)

print("Per-feature time-series plots saved:", len(feature_timeseries_plot_paths))

---

## Cluster Numeric Features by Correlation Distance

Here I use correlation distance and agglomerative clustering to group numeric features into broader similarity clusters.

This is another way of organizing the Silver feature space. Instead of ranking features by abnormal shift, this section groups features by how similarly they tend to move under normal behavior. That can be helpful for subsystem thinking, redundancy checks, and later feature interpretation.

In [ ]:


def build_correlation_cluster_table(
    correlation_matrix: pd.DataFrame,
    *,
    feature_names: Sequence[str],
    cluster_count: Optional[int] = None,
) -> pd.DataFrame:
    """
    Cluster features from a correlation matrix using agglomerative clustering.
    """
    selected_features = [
        feature_name
        for feature_name in feature_names
        if feature_name in correlation_matrix.index and feature_name in correlation_matrix.columns
    ]

    if len(selected_features) < 2:
        return pd.DataFrame(columns=["feature", "cluster_id"])

    cluster_matrix = correlation_matrix.loc[selected_features, selected_features].copy().fillna(0.0)
    distance_matrix = 1.0 - cluster_matrix.abs()

    if cluster_count is None:
        cluster_count = 6 if len(selected_features) >= 6 else max(2, len(selected_features))

    model = AgglomerativeClustering(
        n_clusters=cluster_count,
        metric="precomputed",
        linkage="average",
    )

    labels = model.fit_predict(distance_matrix.values)

    cluster_rows = [
        {
            "feature": feature_name,
            "cluster_id": int(cluster_label),
        }
        for feature_name, cluster_label in zip(selected_features, labels)
    ]

    return (
        pd.DataFrame(cluster_rows)
        .sort_values(["cluster_id", "feature"])
        .reset_index(drop=True)
    )



In [ ]:
cluster_table = build_correlation_cluster_table(
    correlation_matrix_normal,
    feature_names=plot_features,
    cluster_count=None,
)

cluster_path = SILVER_EDA_ARTIFACTS_PATH / "clusters__correlation_agglomerative.csv"
cluster_table.to_csv(cluster_path, index=False)
wandb.save(str(cluster_path))

display(cluster_table.head(30))

----

## Detect and Align Anomaly Onsets

This section finds the start points of anomaly periods and then aligns windows around those onsets.

The main idea here is to answer a timing question:

**what do the key features tend to do right before and right after an anomaly begins?**

By aligning multiple onset windows together, I can look for repeated pre-failure or onset behavior patterns instead of only looking at one episode at a time.

In [ ]:
def find_anomaly_onsets(dataframe: pd.DataFrame) -> pd.DataFrame:
    if "anomaly_flag" not in dataframe.columns:
        return pd.DataFrame(columns=["meta__asset_id", "meta__run_id", "time_index", "event_step"])

    grouping_columns = []
    if "meta__asset_id" in dataframe.columns:
        grouping_columns.append("meta__asset_id")
    if "meta__run_id" in dataframe.columns:
        grouping_columns.append("meta__run_id")

    working = dataframe.copy()

    if "event_step" not in working.columns and "time_index" in working.columns:
        working["event_step"] = working["time_index"]

    if "time_index" not in working.columns:
        working["time_index"] = np.arange(len(working), dtype=np.int64)

    # Ensure sorted within group for consistent onset detection
    if len(grouping_columns) > 0:
        working = working.sort_values(grouping_columns + ["event_step"]).reset_index(drop=True)
        shifted = working.groupby(grouping_columns, dropna=False)["anomaly_flag"].shift(1)
    else:
        working = working.sort_values(["event_step"]).reset_index(drop=True)
        shifted = working["anomaly_flag"].shift(1)

    onset_mask = (working["anomaly_flag"] == 1) & (shifted.fillna(0) == 0)
    onsets = working.loc[onset_mask, grouping_columns + ["time_index", "event_step"]].copy()
    return onsets.reset_index(drop=True)


def sample_onsets_evenly(onsets: pd.DataFrame, max_count: int) -> pd.DataFrame:
    if len(onsets) <= max_count:
        return onsets
    indices = np.linspace(0, len(onsets) - 1, num=max_count)
    indices = [int(round(value)) for value in indices]
    indices = sorted(list(set(indices)))
    return onsets.iloc[indices].reset_index(drop=True)


In [ ]:

onsets_table = find_anomaly_onsets(dataframe)
onsets_table = sample_onsets_evenly(onsets_table, MAX_ONSETS_TO_USE)

onsets_path = SILVER_EDA_ARTIFACTS_PATH / "anomaly_onsets__table.csv"
onsets_table.to_csv(onsets_path, index=False)
wandb.save(str(onsets_path))

logger.info("Anomaly onsets found: %d", len(onsets_table))

# Choose features to align (use TOP_FEATURES if you already computed them; otherwise fallback)
aligned_features = []
for feature_name in TOP_FEATURES if "TOP_FEATURES" in globals() else FEATURE_COLUMNS:
    if feature_name in dataframe.columns and pd.api.types.is_numeric_dtype(dataframe[feature_name]):
        aligned_features.append(feature_name)
    if len(aligned_features) >= 6:
        break

if len(aligned_features) == 0 or len(onsets_table) == 0:
    logger.info("Skipping anomaly-onset alignment (no onsets or no numeric aligned features).")
else:
    relative_steps = np.arange(-PRE_WINDOW_STEPS, POST_WINDOW_STEPS + 1, dtype=np.int64)

    # Build aligned matrices: one matrix per feature, rows = onsets, cols = relative steps
    aligned_mean_rows = []

    for feature_name in aligned_features:
        aligned_values = []

        for onset_index in range(len(onsets_table)):
            onset_row = onsets_table.iloc[onset_index]

            # Filter to the onset's group if asset/run exist
            subset = dataframe
            if "meta__asset_id" in onsets_table.columns:
                subset = subset[subset["meta__asset_id"] == onset_row["meta__asset_id"]]
            if "meta__run_id" in onsets_table.columns:
                subset = subset[subset["meta__run_id"] == onset_row["meta__run_id"]]

            # Ensure ordering by event_step
            if "event_step" in subset.columns:
                subset = subset.sort_values("event_step")
            else:
                subset = subset.sort_values("time_index")

            onset_step = int(onset_row["event_step"])
            start_step = onset_step - PRE_WINDOW_STEPS
            end_step = onset_step + POST_WINDOW_STEPS

            window = subset[(subset["event_step"] >= start_step) & (subset["event_step"] <= end_step)].copy()
            if len(window) == 0:
                continue

            # Reindex to complete relative step range (so we can average across onsets)
            window["relative_step"] = window["event_step"].astype(int) - onset_step
            window = window.set_index("relative_step")

            # Build full aligned vector with NaNs where missing
            aligned_vector = pd.Series(index=relative_steps, dtype=float)
            feature_series = window[feature_name].astype(float)

            # Fill aligned_vector values
            for step_value in feature_series.index:
                if int(step_value) in aligned_vector.index:
                    aligned_vector.loc[int(step_value)] = float(feature_series.loc[step_value])

            # Normalize within-window (z-score) after filling with median to avoid NaN explosion
            filled = aligned_vector.copy()
            median_value = float(np.nanmedian(filled.values))
            filled = filled.fillna(median_value)
            normalized = z_score(filled)

            aligned_values.append(normalized.values)

        if len(aligned_values) == 0:
            continue

        aligned_matrix = np.vstack(aligned_values)
        mean_curve = aligned_matrix.mean(axis=0)

        aligned_mean_rows.append({
            "feature": feature_name,
            "onsets_used": int(aligned_matrix.shape[0]),
        })

        # Plot mean curve
        plt.figure(figsize=(10, 4))
        plt.plot(relative_steps, mean_curve, label=f"{feature_name} (mean)")
        plt.axvline(0, linestyle="--")  # onset marker
        plt.title(f"Anomaly-onset aligned mean (z-scored): {feature_name}")
        plt.xlabel("Relative step (0 = anomaly onset)")
        plt.ylabel("Z-score")
        plt.legend()
        plt.tight_layout()

        plot_path = SILVER_EDA_ARTIFACTS_PATH / f"aligned_onset__mean__{feature_name}.png"
        plt.savefig(plot_path, dpi=200)
        plt.show()
        wandb.save(str(plot_path))

    aligned_summary = pd.DataFrame(aligned_mean_rows)
    aligned_summary_path = SILVER_EDA_ARTIFACTS_PATH / "aligned_onset__summary.csv"
    aligned_summary.to_csv(aligned_summary_path, index=False)
    wandb.save(str(aligned_summary_path))

aligned_features, len(onsets_table)

### Ask

Why is onset alignment useful in this project?

### Answer

Because this project is not only about separating normal rows from abnormal rows after the fact. It is also about early warning behavior.

If certain features tend to move in a repeatable way just before or right at anomaly onset, that pattern can support the later Gold anomaly-screening logic and the project’s early-warning argument.

So this section is helping me look for repeated transition behavior, not just state differences.

----

## Build a PCA View of the Numeric Feature Space

This section creates a two-dimensional PCA projection from the numeric Silver features.

The goal is not to build a final model here. It is to get a quick structure view of the feature space and see whether normal and abnormal rows show any visible separation after basic scaling and imputation.

In [ ]:
def build_pca_projection_artifacts(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    artifacts_dir: Path,
    anomaly_flag_column: str = "anomaly_flag",
    sample_row_count: int = 20000,
    use_robust_scaler: bool = True,
) -> dict:
    """
    Build PCA table and scatter plot artifacts.
    """
    numeric_feature_columns = [
        feature_name
        for feature_name in feature_columns
        if feature_name in dataframe.columns and pd.api.types.is_numeric_dtype(dataframe[feature_name])
    ]

    if len(numeric_feature_columns) == 0:
        return {
            "numeric_feature_columns": [],
            "pca_dataframe": pd.DataFrame(),
            "pca_table_path": None,
            "pca_plot_path": None,
            "explained_variance_ratio": [],
        }

    modeling_columns = numeric_feature_columns + ([anomaly_flag_column] if anomaly_flag_column in dataframe.columns else [])
    modeling_frame = dataframe[modeling_columns].copy()

    if len(modeling_frame) > sample_row_count:
        modeling_frame = modeling_frame.sample(
            n=sample_row_count,
            random_state=42,
        ).reset_index(drop=True)

    feature_matrix = modeling_frame[numeric_feature_columns].copy()

    for feature_name in numeric_feature_columns:
        median_value = float(feature_matrix[feature_name].median(skipna=True))
        feature_matrix[feature_name] = feature_matrix[feature_name].fillna(median_value)

    scaler = RobustScaler() if use_robust_scaler else StandardScaler()
    scaled_matrix = scaler.fit_transform(feature_matrix.values)

    pca_model = PCA(n_components=2, random_state=42)
    pca_values = pca_model.fit_transform(scaled_matrix)

    pca_dataframe = pd.DataFrame(
        {
            "pca_1": pca_values[:, 0],
            "pca_2": pca_values[:, 1],
        }
    )

    if anomaly_flag_column in modeling_frame.columns:
        pca_dataframe[anomaly_flag_column] = (
            pd.to_numeric(modeling_frame[anomaly_flag_column], errors="coerce")
            .fillna(0)
            .astype(int)
            .values
        )
    else:
        pca_dataframe[anomaly_flag_column] = 0

    pca_table_path = artifacts_dir / "pca_2d__table.csv"
    pca_dataframe.to_csv(pca_table_path, index=False)

    plt.figure(figsize=(8, 6))
    normal_points = pca_dataframe[pca_dataframe[anomaly_flag_column] == 0]
    abnormal_points = pca_dataframe[pca_dataframe[anomaly_flag_column] == 1]

    plt.scatter(normal_points["pca_1"], normal_points["pca_2"], s=10, label="Normal")
    if len(abnormal_points) > 0:
        plt.scatter(abnormal_points["pca_1"], abnormal_points["pca_2"], s=20, marker="x", label="Abnormal")

    explained = pca_model.explained_variance_ratio_
    plt.title(f"PCA 2D projection (explained var: {explained[0]:.2f}, {explained[1]:.2f})")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.legend()
    plt.tight_layout()

    pca_plot_path = artifacts_dir / "pca_2d__scatter.png"
    plt.savefig(pca_plot_path, dpi=200)
    plt.show()
    plt.close()

    return {
        "numeric_feature_columns": numeric_feature_columns,
        "pca_dataframe": pca_dataframe,
        "pca_table_path": str(pca_table_path),
        "pca_plot_path": str(pca_plot_path),
        "explained_variance_ratio": [float(explained[0]), float(explained[1])],
        "scaler_name": "RobustScaler" if use_robust_scaler else "StandardScaler",
    }

In [ ]:
pca_artifacts = build_pca_projection_artifacts(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    anomaly_flag_column="anomaly_flag",
    sample_row_count=PCA_SAMPLE_ROW_COUNT,
    use_robust_scaler=USE_ROBUST_SCALER,
)

numeric_feature_columns = pca_artifacts["numeric_feature_columns"]
pca_dataframe = pca_artifacts["pca_dataframe"]
pca_table_path = pca_artifacts["pca_table_path"]
pca_plot_path = pca_artifacts["pca_plot_path"]

if pca_table_path is not None:
    wandb.save(str(pca_table_path))
if pca_plot_path is not None:
    wandb.save(str(pca_plot_path))

if len(pca_dataframe) > 0:
    ledger.add(
        kind="step",
        step="pca_2d_projection",
        message="Computed PCA(2) projection for separation check.",
        data={
            "numeric_feature_count": int(len(numeric_feature_columns)),
            "rows_used": int(len(pca_dataframe)),
            "scaler": pca_artifacts["scaler_name"],
            "explained_variance_ratio": pca_artifacts["explained_variance_ratio"],
            "pca_table_path": pca_table_path,
            "pca_plot_path": pca_plot_path,
        },
        logger=logger,
    )

### Ask

How should I interpret the PCA plot?

### Answer

I should treat it as a quick structure check, not as proof that the classes are perfectly separable.

If normal and abnormal rows show some visible separation, that suggests the feature space contains useful signal. If they overlap heavily, that does not automatically mean the features are useless. It may just mean the separation is more local, nonlinear, or dependent on time context.

So PCA here is a helpful summary view, not the final decision-maker.

---

## Compare Simple Imputation Strategies

This section compares several explainable imputation strategies across the numeric feature set:
- global median
- global mean
- forward fill within ordered groups, then median fallback

The goal is not to declare one strategy universally perfect. The goal is to see which approach best matches the structure of this dataset while staying simple and explainable.

In [ ]:
def compare_imputation_strategies(
    dataframe: pd.DataFrame,
    *,
    feature_columns: Sequence[str],
    sample_row_count: int = 20000,
    artifacts_dir: Path,
    logger,
    ledger=None,
    wandb_run=None,
) -> dict:
    """
    Compare simple imputation strategies for numeric feature columns and
    return both the comparison table and recommendation payload.

    Strategies compared:
    - global median
    - global mean
    - forward fill within (asset, run) ordered by event_step, then median

    Returns a payload containing:
    - numeric_feature_columns
    - comparison_table
    - comparison_csv_path
    - recommendation_payload
    - recommendation_json_path
    """
    numeric_feature_columns = [
        feature_name
        for feature_name in feature_columns
        if feature_name in dataframe.columns
        and pd.api.types.is_numeric_dtype(dataframe[feature_name])
    ]

    if len(numeric_feature_columns) == 0:
        logger.info("No numeric feature columns available for imputation comparison.")
        return {
            "numeric_feature_columns": [],
            "comparison_table": pd.DataFrame(),
            "comparison_csv_path": None,
            "recommendation_payload": {
                "recommendation": None,
                "reason": "No numeric feature columns available for imputation comparison.",
                "has_grouping_columns": False,
                "has_event_step": False,
                "comparison_csv": None,
            },
            "recommendation_json_path": None,
        }

    working_columns = numeric_feature_columns.copy()
    if "meta__asset_id" in dataframe.columns:
        working_columns.append("meta__asset_id")
    if "meta__run_id" in dataframe.columns:
        working_columns.append("meta__run_id")
    if "event_step" in dataframe.columns:
        working_columns.append("event_step")

    working = dataframe[working_columns].copy()

    if len(working) > sample_row_count:
        working = working.sample(n=sample_row_count, random_state=42).reset_index(drop=True)

    # Strategy A: global median
    median_imputed = working[numeric_feature_columns].copy()
    for feature_name in numeric_feature_columns:
        median_value = float(median_imputed[feature_name].median(skipna=True))
        median_imputed[feature_name] = median_imputed[feature_name].fillna(median_value)

    # Strategy B: global mean
    mean_imputed = working[numeric_feature_columns].copy()
    for feature_name in numeric_feature_columns:
        mean_value = float(mean_imputed[feature_name].mean(skipna=True))
        mean_imputed[feature_name] = mean_imputed[feature_name].fillna(mean_value)

    # Strategy C: forward fill within group, then median
    ffill_imputed = working.copy()
    grouping_columns = []

    if "meta__asset_id" in ffill_imputed.columns:
        grouping_columns.append("meta__asset_id")
    if "meta__run_id" in ffill_imputed.columns:
        grouping_columns.append("meta__run_id")

    if len(grouping_columns) > 0 and "event_step" in ffill_imputed.columns:
        ffill_imputed = ffill_imputed.sort_values(grouping_columns + ["event_step"]).reset_index(drop=True)
        for feature_name in numeric_feature_columns:
            ffill_imputed[feature_name] = (
                ffill_imputed.groupby(grouping_columns, dropna=False)[feature_name].ffill()
            )

    for feature_name in numeric_feature_columns:
        median_value = float(ffill_imputed[feature_name].median(skipna=True))
        ffill_imputed[feature_name] = ffill_imputed[feature_name].fillna(median_value)

    comparison_rows = []

    for feature_name in numeric_feature_columns:
        original_series = working[feature_name]
        missing_mask = original_series.isna()

        missing_count = int(missing_mask.sum())
        if missing_count == 0:
            continue

        filled_median = median_imputed.loc[missing_mask, feature_name].astype(float)
        filled_mean = mean_imputed.loc[missing_mask, feature_name].astype(float)
        filled_ffill = ffill_imputed.loc[missing_mask, feature_name].astype(float)

        non_missing = original_series.dropna().astype(float)
        baseline_median = float(non_missing.median()) if len(non_missing) > 0 else 0.0

        median_abs_shift = float((filled_median - baseline_median).abs().mean())
        mean_abs_shift = float((filled_mean - baseline_median).abs().mean())
        ffill_abs_shift = float((filled_ffill - baseline_median).abs().mean())

        comparison_rows.append(
            {
                "feature": feature_name,
                "missing_count": missing_count,
                "missing_percent": float((missing_count / len(working)) * 100.0),
                "baseline_median": baseline_median,
                "median_impute_mean_abs_shift_from_median": median_abs_shift,
                "mean_impute_mean_abs_shift_from_median": mean_abs_shift,
                "ffill_then_median_mean_abs_shift_from_median": ffill_abs_shift,
            }
        )

    if len(comparison_rows) > 0:
        impute_compare_table = (
            pd.DataFrame(comparison_rows)
            .sort_values("missing_percent", ascending=False)
            .reset_index(drop=True)
        )
    else:
        impute_compare_table = pd.DataFrame(
            columns=[
                "feature",
                "missing_count",
                "missing_percent",
                "baseline_median",
                "median_impute_mean_abs_shift_from_median",
                "mean_impute_mean_abs_shift_from_median",
                "ffill_then_median_mean_abs_shift_from_median",
            ]
        )

    impute_compare_path = artifacts_dir / "imputation__comparison.csv"
    impute_compare_table.to_csv(impute_compare_path, index=False)

    if wandb_run is not None:
        wandb.save(str(impute_compare_path))

    has_groups = ("meta__asset_id" in dataframe.columns) and ("meta__run_id" in dataframe.columns)
    has_event_step = "event_step" in dataframe.columns

    if has_groups and has_event_step:
        recommendation = "forward_fill_within_group_then_median"
        recommendation_reason = (
            "Dataset has (asset,run) grouping and event_step ordering; forward fill preserves short gaps "
            "in time series while median handles leading gaps and long missing runs."
        )
    else:
        recommendation = "global_median"
        recommendation_reason = (
            "Dataset does not clearly support within-group forward fill; global median is robust to outliers "
            "and stable for Isolation Forest and One-Class SVM."
        )

    recommendation_payload = {
        "recommendation": recommendation,
        "reason": recommendation_reason,
        "has_grouping_columns": bool(has_groups),
        "has_event_step": bool(has_event_step),
        "comparison_csv": str(impute_compare_path),
    }

    recommendation_json_path = artifacts_dir / "imputation__recommendation.json"
    save_json(
        recommendation_payload,
        file_path=artifacts_dir,
        file_name="imputation__recommendation.json",
    )

    if wandb_run is not None:
        wandb.save(str(recommendation_json_path))

    if ledger is not None:
        ledger.add(
            kind="decision",
            step="imputation_recommendation",
            message="Compared basic imputation strategies and recorded recommendation for Gold layer.",
            data=recommendation_payload,
            logger=logger,
        )

    return {
        "numeric_feature_columns": numeric_feature_columns,
        "comparison_table": impute_compare_table,
        "comparison_csv_path": str(impute_compare_path),
        "recommendation_payload": recommendation_payload,
        "recommendation_json_path": str(recommendation_json_path),
    }

In [ ]:
imputation_artifacts = compare_imputation_strategies(
    dataframe,
    feature_columns=FEATURE_COLUMNS,
    sample_row_count=IMPUTE_SAMPLE_ROW_COUNT,
    artifacts_dir=SILVER_EDA_ARTIFACTS_PATH,
    logger=logger,
    ledger=ledger,
    wandb_run=wandb_run,
)

impute_compare_table = imputation_artifacts["comparison_table"]
impute_compare_path = imputation_artifacts["comparison_csv_path"]
recommendation_payload = imputation_artifacts["recommendation_payload"]
recommendation_json_path = imputation_artifacts["recommendation_json_path"]
numeric_feature_columns = imputation_artifacts["numeric_feature_columns"]

display(impute_compare_table.head(15))
print("Recommendation payload:", recommendation_payload)

### Ask

What am I really trying to learn from this imputation comparison?

### Answer

I want to know which simple strategy makes the most sense for this dataset’s structure.

For a time-ordered grouped dataset like this one, forward fill within the correct group may preserve short gaps better than a pure global fill. But I still want a fallback method for leading gaps or cases where that grouped structure is not enough.

So this comparison is helping me justify a practical Gold imputation choice instead of choosing one by habit.

---

## Finalize the Silver EDA Ledger and Save the Silver EDA Truth Record

Now I wrap the notebook into a formal Silver EDA artifact set.

This section does several important things:
- saves the ledger
- builds the Silver EDA truth record
- records runtime facts
- records the paths to the generated artifacts
- saves the truth record
- appends the truth index
- closes the W&B run

At this point, the EDA notebook output becomes more than just charts and tables inside the notebook. It becomes a tracked pipeline artifact.

In [ ]:
ledger_path = SILVER_EDA_ARTIFACTS_PATH / f"ledger__{DATASET_NAME}__{STAGE}.json"

silver_eda_dataset_name = (
    str(DATASET_NAME).strip().lower()
    if DATASET_NAME is not None
    else str(silver_truth.get("dataset_name", "pump")).strip().lower()
)

silver_eda_process_run_id = (
    SILVER_PROCESS_RUN_ID
    if "SILVER_PROCESS_RUN_ID" in globals()
    else make_process_run_id("silver_eda_process")
)

silver_eda_truth_layer_name = "silver_eda"

truth_config_snapshot = (
    TRUTH_CONFIG
    if "TRUTH_CONFIG" in globals()
    else {
        "runtime": {
            "stage": "silver_eda",
            "dataset": silver_eda_dataset_name,
            "mode": RUN_MODE if "RUN_MODE" in globals() else None,
            "profile": CONFIG_PROFILE if "CONFIG_PROFILE" in globals() else "default",
        }
    }
)

silver_eda_artifact_files = sorted(
    str(path)
    for path in SILVER_EDA_ARTIFACTS_PATH.glob("*")
    if path.is_file()
)

silver_eda_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=silver_eda_dataset_name,
    layer_name=silver_eda_truth_layer_name,
    process_run_id=silver_eda_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=SILVER_TRUTH_HASH,
)

silver_eda_truth = update_truth_section(
    silver_eda_truth,
    "config_snapshot",
    truth_config_snapshot,
)

silver_eda_truth = update_truth_section(
    silver_eda_truth,
    "runtime_facts",
    {
        "input_row_count": int(len(dataframe)),
        "input_column_count": int(dataframe.shape[1]),
        "feature_column_count": int(len(FEATURE_COLUMNS)) if "FEATURE_COLUMNS" in globals() else 0,
        "top_feature_count": int(len(TOP_FEATURES)) if "TOP_FEATURES" in globals() else 0,
        "numeric_feature_count": int(len(numeric_feature_columns)) if "numeric_feature_columns" in globals() else 0,
        "selected_plot_features": plot_features if "plot_features" in globals() else [],
        "state_normalization": (
            state_normalization_payload
            if "state_normalization_payload" in globals()
            else None
        ),
        "episode_status_state_stats": (
            EPISODE_STATUS_STATE_STATS_PAYLOAD
            if "EPISODE_STATUS_STATE_STATS_PAYLOAD" in globals()
            else None
        ),
        "max_onsets_used": int(MAX_ONSETS_TO_USE) if "MAX_ONSETS_TO_USE" in globals() else None,
        "pca_sample_row_count": int(PCA_SAMPLE_ROW_COUNT) if "PCA_SAMPLE_ROW_COUNT" in globals() else None,
        "impute_sample_row_count": int(IMPUTE_SAMPLE_ROW_COUNT) if "IMPUTE_SAMPLE_ROW_COUNT" in globals() else None,
        "parent_truth_hash": SILVER_TRUTH_HASH,
    },
)

artifact_paths_payload = {
    "silver_truth_path": str(SILVER_TRUTH_PATH),
    "silver_eda_artifacts_dir": str(SILVER_EDA_ARTIFACTS_PATH),
    "silver_eda_ledger_path": str(ledger_path),
    "silver_eda_output_files": silver_eda_artifact_files,

    # summary tables
    "effect_size_table_path": str(effect_path) if "effect_path" in globals() else None,
    "episode_status_counts_path": str(episode_status_counts_path) if "episode_status_counts_path" in globals() else None,
    "correlation_matrix_normal_path": str(correlation_matrix_normal_path) if "correlation_matrix_normal_path" in globals() else None,
    "sensor_correlation_pairs_normal_path": str(sensor_correlation_pairs_normal_path) if "sensor_correlation_pairs_normal_path" in globals() else None,
    "sensor_group_map_normal_path": str(sensor_group_map_normal_path) if "sensor_group_map_normal_path" in globals() else None,
    "sensor_fault_pairings_normal_path": str(sensor_fault_pairings_normal_path) if "sensor_fault_pairings_normal_path" in globals() else None,
    "cluster_table_path": str(cluster_path) if "cluster_path" in globals() else None,
    "anomaly_onsets_path": str(onsets_path) if "onsets_path" in globals() else None,
    "pca_table_path": str(pca_table_path) if "pca_table_path" in globals() else None,
    "pca_plot_path": str(pca_plot_path) if "pca_plot_path" in globals() else None,

    # state profiles from the loop-based export
    "feature_profile_normal_path": (
        feature_profile_paths.get("normal")
        if "feature_profile_paths" in globals() and isinstance(feature_profile_paths, dict)
        else None
    ),
    "feature_profile_abnormal_path": (
        feature_profile_paths.get("abnormal")
        if "feature_profile_paths" in globals() and isinstance(feature_profile_paths, dict)
        else None
    ),
    "feature_profile_recovery_path": (
        feature_profile_paths.get("recovery")
        if "feature_profile_paths" in globals() and isinstance(feature_profile_paths, dict)
        else None
    ),

    # plots
    "correlation_heatmap_path": str(correlation_heatmap_path) if "correlation_heatmap_path" in globals() and correlation_heatmap_path is not None else None,
    "top_feature_overlay_path": str(overlay_path) if "overlay_path" in globals() and overlay_path is not None else None,
    "distribution_plot_paths": distribution_plot_paths if "distribution_plot_paths" in globals() else [],
    "feature_timeseries_plot_paths": feature_timeseries_plot_paths if "feature_timeseries_plot_paths" in globals() else [],
}

if "dropped_profile_paths" in globals() and isinstance(dropped_profile_paths, dict):
    artifact_paths_payload.update(dropped_profile_paths)

artifact_paths_payload = {
    key: value
    for key, value in artifact_paths_payload.items()
    if value is not None
}

silver_eda_truth = update_truth_section(
    silver_eda_truth,
    "artifact_paths",
    artifact_paths_payload,
)

silver_eda_truth_record = build_truth_record(
    truth_base=silver_eda_truth,
    row_count=len(dataframe),
    column_count=dataframe.shape[1],
    meta_columns=identify_meta_columns(dataframe),
    feature_columns=identify_feature_columns(dataframe),
)

SILVER_EDA_TRUTH_HASH = silver_eda_truth_record["truth_hash"]

silver_eda_truth_path = save_truth_record(
    silver_eda_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=silver_eda_dataset_name,
    layer_name=silver_eda_truth_layer_name,
)

append_truth_index(
    silver_eda_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

ledger.add(
    kind="step",
    step="finalize",
    message="Saved Silver EDA ledger and truth record.",
    data={
        "ledger_path": str(ledger_path),
        "silver_eda_truth_hash": str(SILVER_EDA_TRUTH_HASH),
        "silver_eda_truth_path": str(silver_eda_truth_path),
    },
    logger=logger,
)

ledger.write_json(ledger_path)

wandb.save(str(ledger_path))
wandb.save(str(silver_eda_truth_path))

logger.info("Silver EDA truth hash: %s", SILVER_EDA_TRUTH_HASH)
logger.info("Silver EDA truth path: %s", silver_eda_truth_path)

print("Silver EDA truth hash:", SILVER_EDA_TRUTH_HASH)
print("Silver EDA truth path:", silver_eda_truth_path)

wandb_run.finish()

----

## Define a Helper to Read Episode-State Summary Details from Truth

This helper function pulls the saved episode-state summary payload back out of the Silver EDA truth record.

The reason I like having this helper is that it makes the saved truth artifact easier to inspect later without having to manually dig through nested JSON every time.

In [ ]:
def pull_episode_status_state_stats_from_truth(truth_record: dict) -> dict:
    runtime_facts = (truth_record or {}).get("runtime_facts", {}) or {}
    artifact_paths = (truth_record or {}).get("artifact_paths", {}) or {}

    payload = runtime_facts.get("episode_status_state_stats", {}) or {}
    episode_status_counts_path = artifact_paths.get("episode_status_counts_path")

    episode_status_counts_records = []
    if episode_status_counts_path:
        path_obj = Path(episode_status_counts_path)
        if path_obj.exists():
            with open(path_obj, "r", encoding="utf-8") as file:
                episode_status_counts_records = json.load(file)

    return {
        "status_column": payload.get("status_column"),
        "episode_column": payload.get("episode_column"),
        "episode_count": int(payload.get("episode_count", 0) or 0),
        "mean_total_rows_per_episode": float(
            payload.get("mean_total_rows_per_episode", 0.0) or 0.0
        ),
        "global_status_stats": payload.get("global_status_stats", []) or [],
        "episode_status_means": payload.get("episode_status_means", []) or [],
        "episode_status_percent_means": payload.get("episode_status_percent_means", []) or [],
        "episode_totals": payload.get("episode_totals", []) or [],
        "episode_status_mean_lookup": payload.get("episode_status_mean_lookup", {}) or {},
        "episode_status_percent_mean_lookup": payload.get("episode_status_percent_mean_lookup", {}) or {},
        "global_status_count_lookup": payload.get("global_status_count_lookup", {}) or {},
        "global_status_percent_lookup": payload.get("global_status_percent_lookup", {}) or {},
        "episode_status_counts_path": episode_status_counts_path,
        "episode_status_counts_records": episode_status_counts_records,
    }

## Review the Saved Episode-State Summary from the Silver EDA Truth Record

After saving the Silver EDA truth record, I pull the episode-state summary back out so I can review the stored defaults and counts directly from the saved artifact.

This is a useful final check because it confirms that the episode-state information I built earlier was actually preserved in the truth record the way I expected.

In [ ]:
episode_status_payload = pull_episode_status_state_stats_from_truth(silver_eda_truth_record)

mean_row_lookup = episode_status_payload.get("episode_status_mean_lookup", {})
mean_percent_lookup = episode_status_payload.get("episode_status_percent_mean_lookup", {})

normal_rows_default = int(round(mean_row_lookup.get("normal", 0)))
abnormal_rows_default = int(round(mean_row_lookup.get("abnormal", 1)))
recovery_rows_default = int(round(mean_row_lookup.get("recovery", 0)))

normal_percent_default = float(mean_percent_lookup.get("normal", 0.0))
abnormal_percent_default = float(mean_percent_lookup.get("abnormal", 0.0))
recovery_percent_default = float(mean_percent_lookup.get("recovery", 0.0))


In [ ]:
episode_status_payload = pull_episode_status_state_stats_from_truth(silver_eda_truth_record)

episode_status_counts_df = pd.DataFrame(
    episode_status_payload["episode_status_counts_records"]
)

display(episode_status_counts_df)

### Ask

What does this final truth-based review confirm?

### Answer

It confirms that the exploratory outputs were not only computed in the notebook, but also preserved in the saved Silver EDA artifact record.

That matters because I want the key EDA findings and support tables to stay traceable and reusable later, especially when they feed into Gold decisions or the project write-up.

So this is both a trust check and a reusability check.

---

## Optional PostgreSQL Write for Silver EDA Outputs

This section is for writing the Silver EDA dataframe into PostgreSQL.

I am treating this as an optional persistence step rather than part of the core Silver EDA deliverable. The main outputs from this notebook are still the saved EDA artifacts, tables, plots, ledger, and Silver EDA truth record. Writing to SQL is useful when I want the Silver EDA artifact available for querying, validation, or downstream integration work.

In [ ]:
SQL_SCHEMAS = {
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "synthetic": "synthetic",
    "truth": "truth",
    "audit": "audit",
}

GOLD_ARTIFACT_TABLES = {
    "baseline_results": "baseline_results",
    "baseline_summary": "baseline_summary",
    "baseline_thresholds": "baseline_thresholds",
    "cascade_results": "cascade_results",
    "cascade_summary": "cascade_summary",
    "cascade_thresholds": "cascade_thresholds",
    "comparison_summary": "comparison_summary",
}

SYNTHETIC_ARTIFACT_TABLES = {
    "batch": "batch",
    "stream": "stream",
    "sensor_messages": "sensor_messages",
}

In [ ]:
SILVER_SCHEMA = SQL_SCHEMAS["silver"]

engine = get_engine_from_env()

silver_sql_dataframe = prepare_layer_dataframe(
    dataframe,
    truth_hash=SILVER_EDA_TRUTH_HASH,
    parent_truth_hash=SILVER_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
    process_run_id=silver_eda_process_run_id,
    add_loaded_at_column=True,
)

silver_table_name = write_layer_dataframe(
    engine=engine,
    dataframe=silver_sql_dataframe,
    schema=SILVER_SCHEMA,
    dataset_name=DATASET_NAME,
    if_exists="replace",
    index=False,
)

print(f"Wrote table: {SILVER_SCHEMA}.{silver_table_name}.eda")

---